<a href="https://colab.research.google.com/github/ts387/cryostar/blob/main/notebooks/CryoSTAR_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CryoSTAR on Google Colab

[CryoSTAR](https://github.com/bytedance/cryostar) recovers the conformational heterogeneity of a protein complex from cryo-EM particle images. It uses a reference atomic model as a structural prior, so it returns both **an ensemble of structures** and **density maps** for the different conformations ([Li *et al.*, *Nature Methods* 2024](https://doi.org/10.1038/s41592-024-02486-1)).

This notebook installs CryoSTAR, gets a dataset (the official tutorial or your own), trains both stages on a single Colab GPU, and shows the results.

| Step | What happens |
|---|---|
| **1 · Setup** | Check the GPU, optionally connect Google Drive, install CryoSTAR |
| **2 · Data** | Download the tutorial or point to your own particles + reference model, then check them |
| **3 · Stage 1 — atom generator** | Learn a latent code for every particle and a coarse-grained structure for each code |
| **4 · Stage 2 — density generator** | Turn the latent codes into density maps |
| **5 · Save** | Download everything as a `.zip` |

### How to use it
1. **Runtime ▸ Change runtime type ▸ T4 GPU** (any GPU works; a faster one finishes sooner).
2. Run the cells from top to bottom with ▶ or **Shift + Enter**. Each cell has a small form; change the values in the form and then run the cell.
3. Cells marked (optional) can be skipped.

> **Colab machines are temporary.** Everything under `/content` disappears when the runtime disconnects. To keep results, tick *Use Google Drive* in step 1.2 or download them in step 5.

## 1 · Setup

In [ ]:
#@title 1.1 · Check the GPU and load helpers { display-mode: "form" }
#@markdown Run this first. It confirms that a GPU is attached and defines the small helper functions used by the cells below.

import os, re, sys, html, json, time, math, shutil, subprocess, zipfile
from pathlib import Path
import numpy as np
from IPython.display import display, HTML, Markdown

IN_COLAB = "google.colab" in sys.modules
CONTENT = Path("/content") if IN_COLAB else Path.cwd()
REPO_DIR = CONTENT / "cryostar"
STAR_DIR = REPO_DIR / "projects" / "star"
DATA_ROOT = CONTENT / "data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Colours for plots (blue = data series, a light-to-dark blue ramp for densities).
SERIES_BLUE = "#2a78d6"
SERIES_ORANGE = "#eb6834"
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

_NOTE_STYLE = {"info": ("#2a78d6", "ℹ️"), "ok": ("#0ca30c", "✅"),
               "warn": ("#fab219", "⚠️"), "error": ("#d03b3b", "❌")}


def note(msg, kind="info"):
    """Show a short coloured message box."""
    color, icon = _NOTE_STYLE[kind]
    display(HTML(f'<div style="border-left:4px solid {color};padding:6px 12px;margin:6px 0;'
                 f'line-height:1.5">{icon} {msg}</div>'))


def need(**names):
    """Stop with a friendly message if a variable from an earlier cell is missing."""
    missing = [cell for name, cell in names.items() if name not in globals()]
    if missing:
        raise RuntimeError("Please run these cells first: " + "; ".join(dict.fromkeys(missing)))


def sh(cmd, cwd=None, quiet=False):
    """Run a shell command, stream its output into the cell and raise if it fails."""
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in proc.stdout:
        out.append(line)
        if not quiet:
            print(line, end="")
    if proc.wait() != 0:
        if quiet:
            print("".join(out[-40:]))
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
    return "".join(out)


def fmt_time(seconds):
    seconds = int(seconds)
    h, m, s = seconds // 3600, seconds % 3600 // 60, seconds % 60
    return f"{h}h {m:02d}m" if h else f"{m}m {s:02d}s"


def style_axes(ax):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#999999")
    ax.tick_params(colors="#52514e", labelsize=9)
    ax.grid(True, color="#e6e6e6", linewidth=0.6)
    ax.set_axisbelow(True)


def blue_cmap():
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list("blue_ramp", BLUE_RAMP)


# ---------- configs ----------
def write_config(path, base, **sections):
    """Write an mmengine config that inherits `base` and overrides only the given keys."""
    def lit(v):
        return repr(str(v)) if isinstance(v, Path) else repr(v)
    lines = [f'_base_ = "./{base}"', ""]
    for name, value in sections.items():
        if isinstance(value, dict):
            body = "".join(f"    {k}={lit(v)},\n" for k, v in value.items())
            lines.append(f"{name} = dict(\n{body})")
        else:
            lines.append(f"{name} = {lit(value)}")
    Path(path).write_text("\n".join(lines) + "\n")
    return Path(path)


# ---------- training runs ----------
STEP_RE = re.compile(r"epoch (\d+) \[(\d+)/(\d+)\]")
WARNING_RE = re.compile(r"^\S+\.py:\d+: |^\[(rank\d+\]:\[)?W\d{4}|💡 Tip|litlogger")
TIMESTAMP_RE = re.compile(r"^\d\d/\d\d \d\d:\d\d:\d\d - ")
ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")


def ensure_work_dirs():
    """CryoSTAR writes to projects/star/work_dirs; point it at Google Drive when enabled."""
    link = STAR_DIR / "work_dirs"
    persist = globals().get("PERSIST_DIR")
    if persist is None:
        if not link.exists():
            link.mkdir()
        return link
    target = Path(persist) / "work_dirs"
    if str(target).startswith("/content/drive/") and not os.path.ismount("/content/drive"):
        raise RuntimeError("Google Drive is no longer connected to this runtime (it was unmounted): run 1.2 again.")
    target.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if Path(os.readlink(link)) == target:
            return link
        link.unlink()
    elif link.exists():  # runs made before Drive was connected: move them over
        for p in link.iterdir():
            shutil.move(str(p), str(target / p.name))
        link.rmdir()
    link.symlink_to(target, target_is_directory=True)
    return link


def _progress_html(title, step, total, rate, last):
    if not total:
        bar, text = '<progress style="width:100%;height:14px"></progress>', "starting…"
    else:
        step = min(step, total)
        bar = f'<progress value="{step}" max="{total}" style="width:100%;height:14px"></progress>'
        text = f"step {step:,} / {total:,} ({100 * step / total:.1f}%)"
        if rate:
            text += f" · about {fmt_time((total - step) / rate)} left"
    return (f'<div style="margin:4px 0"><b>{title}</b> — {text}<br>{bar}'
            f'<div style="font-family:monospace;font-size:12px;opacity:.75">{last}</div></div>')


def run_stage(title, script, config_path, max_steps=None, max_epochs=None):
    """Run a CryoSTAR training script, stream its log and show a progress bar. Returns the run folder."""
    work = ensure_work_dirs()
    before = set(os.listdir(work))
    rel_cfg = Path(config_path).relative_to(STAR_DIR)
    print(f"$ cd {STAR_DIR} && python {script} {rel_cfg}\n")
    status = display(HTML(_progress_html(title, 0, max_steps, None, "")), display_id=True)
    proc = subprocess.Popen([sys.executable, script, str(rel_cfg)], cwd=STAR_DIR,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines, hidden, bad_images, shown = [], 0, 0, 0
    skip_next = collapsing = False
    total, first, rc = max_steps, None, None
    t_start = time.time()
    try:
        for line in proc.stdout:
            lines.append(line)
            plain = ANSI_RE.sub("", line)
            message = plain.split("INFO - ", 1)[-1].strip()
            m = STEP_RE.search(plain)
            if m:
                epoch, batch, n_batches = map(int, m.groups())
                if total is None and max_epochs:
                    total = max_epochs * n_batches
                step = epoch * n_batches + batch
                first = first or (step, time.time())
                elapsed = time.time() - first[1]
                rate = (step - first[0]) / elapsed if elapsed > 0 and step > first[0] else None
                status.update(HTML(_progress_html(title, step, total, rate, html.escape(message))))
            if "Particle image" in plain and "invalid" in plain:
                bad_images += 1
            # Keep the cell readable: fold the full config / model printouts and library warnings.
            if collapsing:
                if not (TIMESTAMP_RE.match(plain) or plain.startswith("Traceback") or "Error" in plain):
                    hidden += 1
                    continue
                collapsing = False
            if WARNING_RE.search(plain):
                hidden, skip_next = hidden + 1, True
                continue
            if skip_next and plain[:1].isspace():
                hidden += 1
                continue
            skip_next = False
            if TIMESTAMP_RE.match(plain) and (message in ("Config:", "Model summary:") or message.startswith("CryoModel(")):
                collapsing = True
                line = line.rstrip("\n").rstrip("(") + " … (folded, see stdout.txt)\n"
            print(line, end="")
            shown = len(lines)
        rc = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
    new = sorted(set(os.listdir(work)) - before)
    run_dir = (work / new[-1]) if new else None
    if run_dir is not None:
        (run_dir / "stdout.txt").write_text("".join(lines))
    if hidden:
        print(f"\n({hidden} lines of warnings and printouts folded; the full output is in {run_dir}/stdout.txt)")
    if bad_images:
        note(f"{bad_images} particle image(s) could not be read and were replaced by zeros. "
             "Check the particle files chosen in step 2.1.", "warn")
    if rc == 0:
        status.update(HTML(_progress_html(title, total or 1, total or 1, None,
                                          f"finished in {fmt_time(time.time() - t_start)}")))
        return run_dir
    if rc is None:
        if run_dir is not None:
            note("Training stopped. Results from evaluations that already finished are still in "
                 f"<code>{run_dir}</code>.", "warn")
        return run_dir
    log = ANSI_RE.sub("", "".join(lines[-200:]))
    if shown < len(lines):  # the end of the output was folded away: show it now
        print("\n──── last lines of the output ────\n" + "".join(log.splitlines(keepends=True)[-30:]))
    hint = ""
    if "out of memory" in log.lower():
        hint = "The GPU ran out of memory: lower BATCH_SIZE or IMAGE_SIZE and run again. "
    elif "size mismatch" in log or "Error(s) in loading state_dict" in log:
        hint = ("The checkpoint in CONTINUE_FROM doesn't fit the network being trained: that run used another "
                "reference model, image size, latent dimension or decoder width. ")
    elif "No such file or directory" in log or "FileNotFoundError" in log:
        hint = "A file could not be found: re-check the paths in step 2. "
    raise RuntimeError(f"Training failed (exit code {rc}). {hint}See the log above"
                       + (f" or {run_dir}/stdout.txt." if run_dir else "."))


def result_dirs(run_dir, marker):
    """Evaluation sub-folders (named <epoch>_<step>) of a run that contain `marker`, oldest first."""
    run_dir = Path(run_dir)
    return sorted(p for p in run_dir.iterdir()
                  if p.is_dir() and re.fullmatch(r"\d{4}_\d{7}", p.name) and (p / marker).exists())


def _same_file(a, b):
    """Same file, or same content (for .star files, ignoring comment lines such as starfile's time stamp)."""
    a, b = Path(str(a)), Path(str(b))
    if a.resolve() == b.resolve():
        return True
    if not (a.is_file() and b.is_file()):
        return False
    if a.suffix == b.suffix == ".star":
        strip = lambda f: [line for line in f.read_text().splitlines() if not line.startswith("#")]
        return strip(a) == strip(b)
    import filecmp
    return filecmp.cmp(a, b, shallow=False)


def data_inputs(*keys):
    """What a training config takes from DATA, to notice when DATA changes after the config was written."""
    return tuple(str(DATA[k]) for k in keys)


def run_folder(path):
    """The training run folder from what was pasted: the run folder itself or one of its result folders."""
    ensure_work_dirs()
    path = Path(str(path).strip()).expanduser()
    if not path.is_dir():
        raise FileNotFoundError(f"{path} is not a folder. Give the full path of the run folder (atom_colab_… or "
                                "density_colab_…); with Google Drive on (1.2) that is "
                                "/content/drive/MyDrive/<DRIVE_FOLDER>/work_dirs/….")
    if not (path / "config.py").exists() and (path.parent / "config.py").exists():
        path = path.parent  # a result folder (<epoch>_<step>) was given
    return path


def run_config(run_dir):
    """The full config a training run saved in its config.py (None if there is none)."""
    cfg = Path(run_dir) / "config.py"
    if not cfg.is_file():
        return None
    from mmengine import Config
    try:
        return Config.fromfile(str(cfg))
    except Exception:
        return None


def run_config_inputs(run_dir):
    """dataset_attr of a training run's config.py ({} if there is none)."""
    cfg = run_config(run_dir)
    return dict(cfg.get("dataset_attr") or {}) if cfg is not None else {}


def continue_point(path, stage):
    """The checkpoint CONTINUE_FROM (3.1, 4.1) points to, as (run folder, ckpt.pt, the run's config). `path` is a run
    folder (its newest checkpoint is used), one of its <epoch>_<step> folders or a ckpt.pt in one."""
    path = Path(str(path).strip()).expanduser()
    if path.name == "ckpt.pt" and not path.is_file():
        raise FileNotFoundError(f"{path} does not exist.")
    if path.is_file() and path.name != "ckpt.pt":
        raise ValueError(f"{path} is not a checkpoint. Give a run folder, one of its <epoch>_<step> folders or a ckpt.pt.")
    given = path.parent if path.is_file() else path
    run = run_folder(given)
    cfg = run_config(run)
    if cfg is None:
        raise FileNotFoundError(f"{run} has no config.py, so it is not a CryoSTAR run folder (atom_… or density_…).")
    kind = "density" if "given_z" in (cfg.get("extra_input_data_attr") or {}) else "atom"
    if kind != stage:
        cell = {"atom": "3.1", "density": "4.1"}[kind]
        raise ValueError(f"{run.name} is a {'stage-2' if kind == 'density' else 'stage-1'} run; continue it in {cell}.")
    # A continuation without a checkpoint of its own can be started again from the run it continued.
    parent = (cfg.get("colab_continue") or {}).get("from_run")
    back = (f" {run.name} continues {parent}: give that folder in CONTINUE_FROM instead, to continue from its newest "
            f"checkpoint ({'MAX_STEPS' if stage == 'atom' else 'MAX_EPOCHS'} = auto trains what is left of its plan)."
            if parent else "")
    if given.resolve() == run.resolve():
        saved = result_dirs(run, "ckpt.pt")
        if not saved:
            why = ("its last checkpoint is missing" if result_dirs(run, "z.npy")
                   else "it stopped before its first evaluation")
            raise FileNotFoundError(f"{run} holds no checkpoint (ckpt.pt): {why}." + back)
        folder = saved[-1]
    else:
        folder = given
        if not re.fullmatch(r"\d{4}_\d{7}", folder.name):
            raise ValueError(f"{folder} is not one of {run.name}'s result folders (named <epoch>_<step>).")
        if not (folder / "ckpt.pt").is_file():
            msg = f"{folder} holds no checkpoint (ckpt.pt)."
            if stage == "atom":
                saved = result_dirs(run, "ckpt.pt")
                msg += (" Stage 1 keeps only the checkpoint of its latest evaluation"
                        + (f" (here in {saved[-1].name}); give the run folder to continue from it." if saved else "."))
            if not result_dirs(run, "ckpt.pt"):
                msg += back
            raise FileNotFoundError(msg)
    return run, (folder / "ckpt.pt").resolve(), cfg


def config_changes(section, base):
    """The entries of a config section (e.g. a run's `model`) that differ from `base`, as plain nested dicts."""
    out = {}
    for k, v in section.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            changed = config_changes(v, base[k])
            if changed:
                out[k] = changed
        elif k not in base or v != base[k]:
            out[k] = v.to_dict() if hasattr(v, "to_dict") else v
    return out


def same_codes(a, b):
    """True if two .npy files hold the same latent codes."""
    za, zb = np.load(a, mmap_mode="r"), np.load(b, mmap_mode="r")
    return za.shape == zb.shape and np.array_equal(za, zb)


def note_gone_inputs(run_dir):
    """Warn about files and the particle folder a run was trained with that no longer exist here (run_inputs_differ
    can't compare them). 2.1's local copy of the particles is named after the source and the contrast, so after a new
    runtime its old folder is back only if INVERT_CONTRAST is unchanged."""
    attr = run_config_inputs(run_dir)
    for k, what in (("starfile_path", ".star file (poses)"), ("ref_pdb_path", "model")):
        if attr.get(k) and not Path(str(attr[k])).is_file():
            note(f"The {what} that {Path(run_dir).name} was trained with (<code>{attr[k]}</code>) is no longer here, "
                 "so it can't be compared with the one selected now. Continue only if 2.1, 2.2 and 2.3 were run with "
                 "the same settings as for that run.", "warn")
    if attr.get("dataset_dir") and not Path(str(attr["dataset_dir"])).is_dir():
        note(f"The particle folder that {Path(run_dir).name} was trained with (<code>{attr['dataset_dir']}</code>) is "
             "no longer here, so its contrast can't be compared with the one selected now. Continue only if "
             "INVERT_CONTRAST in 2.1 is set as it was for that run.", "warn")


def inverted_copy(folder):
    """True if `folder` is a local copy of particles that 2.1 made with inverted contrast."""
    manifest = Path(str(folder)) / ".sources.json"
    try:
        return bool(json.loads(manifest.read_text()).get("__invert__")) if manifest.exists() else False
    except ValueError:
        return False


def run_inputs_differ(run_dir):
    """What a training run used that differs from the dataset selected now, as [(what, then, now)].

    Files the run used that no longer exist here can't be compared and are left out."""
    attr = run_config_inputs(run_dir)
    if not attr or "DATA" not in globals():
        return []
    out = []
    for k, key, what in (("starfile_path", "star", "set of poses"), ("ref_pdb_path", "pdb", "model")):
        then, now = Path(str(attr.get(k) or "")), Path(str(DATA[key]))
        if attr.get(k) and then.is_file() and not _same_file(then, now):
            same_name = then.name == now.name
            out.append((what, str(then) if same_name else then.name, str(now) if same_name else now.name))
    if attr.get("apix") and abs(float(attr["apix"]) - float(DATA["apix"])) > 1e-4:
        out.append(("pixel size", f"{float(attr['apix']):.4f} Å", f"{float(DATA['apix']):.4f} Å"))
    if attr.get("dataset_dir") and Path(str(attr["dataset_dir"])).is_dir():
        then_inv, now_inv = inverted_copy(attr["dataset_dir"]), bool(DATA.get("inverted_contrast"))
        if then_inv != now_inv:
            label = {True: "inverted", False: "as stored"}
            out.append(("particle contrast (INVERT_CONTRAST)", label[then_inv], label[now_inv]))
    finished = result_dirs(run_dir, "z.npy")
    if finished:
        n = len(np.load(finished[-1] / "z.npy", mmap_mode="r"))
        if n != DATA["n"]:
            out.append(("number of particles", f"{n:,}", f"{DATA['n']:,}"))
    return out


def note_run_inputs(run_dir, consequence, ignore=()):
    diffs = [d for d in run_inputs_differ(run_dir) if d[0] not in ignore]
    for what, then, now in diffs:
        note(f"This run was trained with a different {what} (<code>{then}</code>) from the one selected now "
             f"(<code>{now}</code>). {consequence} Run 2.1, 2.2 and 2.3 again with the settings you trained "
             "with (see Troubleshooting).", "error")
    return diffs


def collect_results(what="final results only"):
    """Files to save from the training runs, as [(file, path in the saved copy)], and the names of training inputs
    that no longer exist here. 5.1 packs them into a .zip, 5.2 copies them to Google Drive."""
    g = globals()
    files = []
    for run_name, result_name in (("ATOM_RUN", "ATOM_RESULT"), ("DENSITY_RUN", "DENSITY_RESULT")):
        if g.get(run_name) is None:
            continue
        run = Path(g[run_name])
        if what == "complete run folders":
            # A run trained further with CONTINUE_FROM is a new folder; the runs it continues hold the steps before it.
            runs = [run]
            while (prev := ((run_config(runs[-1]) or {}).get("colab_continue") or {}).get("from_run")) \
                    and Path(str(prev)).is_dir() and Path(str(prev)).name not in [r.name for r in runs]:
                runs.append(Path(str(prev)))
            files += [(f, f.relative_to(r.parent)) for r in runs for f in r.rglob("*") if f.is_file()]
            continue
        chosen = [f for f in run.iterdir() if f.is_file()]
        if g.get(result_name) is not None:
            chosen += [f for f in Path(g[result_name]).rglob("*") if f.is_file()]
        files += [(f, f.relative_to(run.parent)) for f in chosen]
    if not files:
        return [], []
    # The model and poses stage 1 was trained with (centred / mirrored by 2.2 and 2.3), from its config.
    attr = run_config_inputs(g["ATOM_RUN"]) if g.get("ATOM_RUN") is not None else {}
    if attr:
        used = [attr.get("ref_pdb_path"), attr.get("starfile_path")]
    else:
        used = [g["DATA"].get("pdb"), g["DATA"].get("star")] if "DATA" in g else []
    files += [(Path(str(f)), Path("inputs") / Path(str(f)).name) for f in used if f and Path(str(f)).is_file()]
    gone = [Path(str(f)).name for f in used if f and not Path(str(f)).is_file()]
    return files, gone


def note_pose_status():
    """Say which poses and model training will use (own data only)."""
    if "DATA" not in globals() or DATA.get("kind") != "own":
        return
    check = DATA.get("pose_check")
    if check == "inverted contrast":
        note("2.3 found that the particles' contrast is inverted. Change INVERT_CONTRAST in 2.1 and run 2.1–2.3 "
             "again before training.", "error")
    elif check is None:
        note("The pose check (2.3) hasn't been run since the dataset was chosen (2.1) or the model was checked (2.2). "
             "If the consensus map or cryoDRGN's volumes were ever hand-flipped, run 2.3 first; otherwise "
             "CryoSTAR's structures and maps will be the mirror image of your model.", "warn")
    elif DATA.get("hand_flip"):
        note(f"Poses mirrored by 2.3 (map flipped along {DATA['hand_flip']}): <code>{Path(DATA['star']).name}</code>, "
             f"model <code>{Path(DATA['pdb']).name}</code>.")
    elif check == "as given":
        note(f"Poses as given (2.3 confirmed their hand), model <code>{Path(DATA['pdb']).name}</code>.")
    else:
        serious = any(w in check for w in ("no match", "weak", "not docked", "negative", "axis unclear", "flip"))
        note(f"Poses as given; 2.3 could not confirm their hand ({check}, see its message). Model "
             f"<code>{Path(DATA['pdb']).name}</code>.", "warn" if serious else "info")


def read_metrics(run_dir):
    """Parse the loss lines of a CryoSTAR log into {metric: ([steps], [values])}."""
    metrics = {}
    for log in sorted(Path(run_dir).glob("*.log")):
        for line in log.read_text().splitlines():
            m = STEP_RE.search(line)
            if not m or " | " not in line:
                continue
            epoch, batch, n_batches = map(int, m.groups())
            step = epoch * n_batches + batch
            for part in line.split(" | ")[1:]:
                key, _, value = part.partition(": ")
                try:
                    metrics.setdefault(key.strip(), ([], []))
                    metrics[key.strip()][1].append(float(value))
                    metrics[key.strip()][0].append(step)
                except ValueError:
                    pass
    return metrics


def split_models(pdb_text):
    """Split a multi-model PDB into a list of single-model PDB strings."""
    models, current = [], []
    for line in pdb_text.splitlines():
        if line.startswith("MODEL"):
            current = []
        elif line.startswith("ENDMDL"):
            models.append("\n".join(current) + "\nEND\n")
        elif line.startswith(("ATOM", "HETATM")):
            current.append(line)
    return models or [pdb_text]


# ---------- cryoDRGN inputs -> RELION .star ----------
# pose.pkl holds (rots, trans). rots[i] is RELION's Euler_angles2matrix(rot, tilt, psi), which is also the
# matrix CryoSTAR builds from a .star. trans is the RELION origin shift as a fraction of the box. ctf.pkl is
# (N, 9): D, Apix (of the original stack), dfU, dfV (Å), dfang (°), kV, Cs (mm), amplitude contrast, phase
# shift (°). The same holds for pkls from parse_pose_star / parse_ctf_star and parse_pose_csparc /
# parse_ctf_csparc. (parse_pose_csparc assumes cryoSPARC shifts are in pixels of the particle blobs; see the
# .cs check in 2.1.)

class RowCountError(ValueError):
    pass


def _load_pkl(path):
    import pickle
    with open(path, "rb") as f:
        return pickle.load(f)


def _as_array(x):
    if hasattr(x, "detach"):  # torch tensor
        x = x.detach().cpu().numpy()
    return np.asarray(x, dtype=np.float64)


def _stack_files(particles):
    """Files behind a .mrcs/.mrc stack or a cryoDRGN .txt list, with image counts, box size and header Å/px."""
    import mrcfile
    p = Path(particles)
    if p.name.lower().endswith((".ft.mrcs", ".ft.txt")):
        raise ValueError(f"{p.name} was written by `cryodrgn preprocess` and holds Fourier-space images. "
                         "Use the original particle stack, or one made with `cryodrgn downsample`.")
    if p.suffix.lower() in (".mrcs", ".mrc"):
        files = [p]
    elif p.suffix.lower() == ".txt":
        lines = [l.strip() for l in p.read_text().splitlines() if l.strip()]
        files = [Path(l) if os.path.isabs(l) else p.parent / l for l in lines]
        if not files:
            raise ValueError(f"{p} lists no particle stacks.")
    elif p.suffix.lower() in (".star", ".cs"):
        raise ValueError(f"{p.name}: give the consolidated particle stack, not the .star/.cs file. For a RELION "
                         ".star choose 'My own data: a RELION or cryoSPARC .star file'; for a .cs, make a stack "
                         "with `cryodrgn downsample <file>.cs --datadir <cryoSPARC project> -D <box> -o particles.mrcs`.")
    else:
        raise ValueError(f"{p.name}: the particle stack must be a .mrcs/.mrc file or a cryoDRGN .txt list.")
    counts, sizes, hdr_apix = [], set(), None
    for f in files:
        if not f.is_file():
            raise FileNotFoundError(f"Particle stack not found: {f}")
        with mrcfile.open(f, mode="r", header_only=True, permissive=True) as m:
            sizes.add((int(m.header.nx), int(m.header.ny)))
            counts.append(max(int(m.header.nz), 1))
            hdr_apix = float(m.voxel_size.x) if hdr_apix is None else hdr_apix
    (nx, ny), = sizes if len(sizes) == 1 else [(0, 1)]
    if len(sizes) != 1 or nx != ny:
        raise ValueError(f"Particle images must be square and all the same size; found {sorted(sizes)}.")
    if nx % 2:
        raise ValueError(f"The particles are {nx} px, an odd size. Old cryoDRGN `preprocess` wrote Fourier-space "
                         "stacks of D+1 px, which can't be used; use the original stack or `cryodrgn downsample` "
                         "output (even box sizes).")
    return files, np.asarray(counts), nx, hdr_apix


def _match_rows(n_stack, n_pose, n_ctf, ind):
    """Rows of the stack, pose.pkl and ctf.pkl that describe the same particles (cryoDRGN's --ind rules)."""
    lens = {"particle stack": n_stack, "pose.pkl": n_pose, "ctf.pkl": n_ctf}
    listing = ", ".join(f"{k} {v:,}" for k, v in lens.items())
    if ind is None:
        if n_stack == n_pose == n_ctf:
            r = np.arange(n_stack)
            return r, r, r, f"stack, pose.pkl and ctf.pkl all have {n_stack:,} rows"
        raise RowCountError(f"The inputs have different numbers of particles ({listing}). If cryoDRGN selected "
                            "particles with an ind.pkl (--ind), give it as IND_PKL.")
    n, n_full = len(ind), max(lens.values())
    if n_full == n:
        if any(v != n for v in lens.values()):
            raise RowCountError(f"The row counts don't fit ind.pkl ({n:,} entries): {listing}.")
        if np.array_equal(ind, np.arange(n)) or ind.max() >= n:
            r = np.arange(n)
            return r, r, r, ("ind.pkl selects every row" if ind.max() < n
                             else f"stack and pkls were already filtered with ind.pkl ({n:,} rows)")
        raise RowCountError(f"Everything has {n:,} rows and ind.pkl only reorders them, so it is unclear whether "
                            "the inputs are already in ind.pkl order. Leave IND_PKL empty if they are.")
    if ind.max() >= n_full:
        raise RowCountError(f"ind.pkl refers to particle {ind.max():,} but the largest input has {n_full:,} rows.")
    rows, parts = {}, []
    for k, v in lens.items():
        if v == n_full:
            rows[k] = ind
            parts.append(f"{k}: {n:,} of {v:,} rows selected with ind.pkl")
        elif v == n:
            rows[k] = np.arange(n)
            parts.append(f"{k}: already filtered ({v:,} rows)")
        else:
            raise RowCountError(f"With ind.pkl ({n:,} entries) each input needs {n_full:,} rows (all particles) "
                                f"or {n:,} rows (already filtered); got {listing}.")
    return rows["particle stack"], rows["pose.pkl"], rows["ctf.pkl"], "; ".join(parts)


def cryodrgn_to_star(particles, pose_pkl, ctf_pkl, out_star, ind_pkl=None, apix=None):
    """Write a RELION 3.1 .star with which CryoSTAR sees the same particles, poses and CTF as cryoDRGN.

    Returns a dict; its "notes" are (kind, message) pairs with kind "warn" or "info".
    """
    import warnings
    import pandas as pd, starfile
    from scipy.spatial.transform import Rotation
    notes = []
    pose = _load_pkl(pose_pkl)
    if isinstance(pose, (tuple, list)) and len(pose) == 2:
        rots, trans = _as_array(pose[0]), _as_array(pose[1])
    elif isinstance(pose, (tuple, list)) and len(pose) != 1:
        raise ValueError(f"{Path(pose_pkl).name} holds {len(pose)} items; a cryoDRGN pose.pkl holds (rotations, shifts).")
    else:
        rots = _as_array(pose[0] if isinstance(pose, (tuple, list)) else pose)
        trans = np.zeros((len(rots), 2))
        notes.append(("warn", "pose.pkl holds rotations only, so all shifts are set to 0."))
    if rots.ndim != 3 or rots.shape[1:] != (3, 3) or trans.shape != (len(rots), 2):
        raise ValueError(f"{Path(pose_pkl).name} doesn't look like a cryoDRGN pose.pkl "
                         f"(rotations {rots.shape}, shifts {trans.shape}).")
    ctf = _as_array(_load_pkl(ctf_pkl))
    if ctf.ndim != 2 or ctf.shape[1] != 9:
        old = " It looks like the 8-column layout of cryoDRGN < 0.1.1; re-run parse_ctf_*." \
            if ctf.ndim == 2 and ctf.shape[1] == 8 else ""
        raise ValueError(f"{Path(ctf_pkl).name} should be an (N, 9) cryoDRGN ctf.pkl, not {ctf.shape}.{old}")
    ind = None
    if ind_pkl:
        ind = np.asarray(_load_pkl(ind_pkl)).reshape(-1)
        ind = np.flatnonzero(ind) if ind.dtype == bool else ind
        if ind.size == 0 or not np.all(np.mod(ind, 1) == 0) or ind.min() < 0:
            raise ValueError(f"{Path(ind_pkl).name} must list particle indices (whole numbers ≥ 0).")
        ind = ind.astype(np.int64)
    files, counts, box, hdr_apix = _stack_files(particles)
    stack_rows, pose_rows, ctf_rows, rows_desc = _match_rows(int(counts.sum()), len(rots), len(ctf), ind)
    rots, trans, ctf = rots[pose_rows], trans[pose_rows], ctf[ctf_rows]
    n = len(stack_rows)

    bad = np.flatnonzero(~(np.isfinite(rots).all(axis=(1, 2)) & np.isfinite(trans).all(axis=1)))
    if len(bad):
        raise ValueError(f"{len(bad):,} particles have NaN poses (e.g. rows {bad[:5].tolist()}). cryoDRGN's "
                         "parse_pose_csparc gives NaN when a cryoSPARC pose is exactly zero. Save an ind.pkl that "
                         "leaves those rows out and give it as IND_PKL.")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # gimbal lock at tilt 0/180: the angles change, the matrix doesn't
        rotation = Rotation.from_matrix(rots.transpose(0, 2, 1))
        proper = rotation.as_matrix().transpose(0, 2, 1)
        euler = rotation.as_euler("ZYZ", degrees=True)  # RELION rot, tilt, psi
    if np.abs(np.linalg.det(rots) - 1).max() > 1e-3 or np.abs(proper - rots).max() > 1e-3:
        raise ValueError("pose.pkl contains matrices that are not rotations.")
    rots = proper
    tmax = float(np.abs(trans).max()) if n else 0.0
    if tmax > 1:
        raise ValueError(f"pose.pkl shifts reach {tmax:.2f} box lengths, so they are in pixels rather than "
                         "fractions of the box (as `cryodrgn abinit` writes them). Divide them by the box size.")
    if tmax > 0.25:
        notes.append(("warn", f"Some shifts are {tmax:.2f} of the box. cryoDRGN stores them as fractions of the "
                              "box; check that this pose.pkl isn't in pixels."))
    if not np.isfinite(ctf[:, :8]).all():
        raise ValueError("ctf.pkl has missing (NaN) defocus, voltage, Cs or amplitude-contrast values.")

    D0, apix0 = float(ctf[0, 0]), float(ctf[0, 1])
    phys = ctf[:, 0] * ctf[:, 1]  # box size in Å; unchanged by downsampling
    if apix:
        apix, apix_from = float(apix), "PIXEL_SIZE"
        if abs(apix * box / phys[0] - 1) > 1e-3:
            notes.append(("warn", f"PIXEL_SIZE {apix:g} Å × {box} px = {apix * box:.1f} Å, but ctf.pkl implies a "
                                  f"{phys[0]:.1f} Å box ({D0:g} px × {apix0:g} Å). Using PIXEL_SIZE."))
    else:
        if np.ptp(phys) > 1e-5 * phys[0]:
            raise ValueError("ctf.pkl mixes particles with different pixel or box sizes (several optics groups); "
                             "CryoSTAR supports one. Enter PIXEL_SIZE to use a single value.")
        apix = float(phys[0]) / box
        apix_from = f"ctf.pkl ({D0:g} px × {apix0:g} Å" + (f", scaled to {box} px)" if box != D0 else ")")
    if hdr_apix and abs(hdr_apix / apix - 1) > 0.01:
        notes.append(("info", f"The stack's MRC header says {hdr_apix:.4f} Å/px; {apix:.4f} Å/px from {apix_from} "
                              "is used. Stacks written by `cryodrgn downsample` from a .cs or .txt always carry a wrong "
                              "header value, so nothing needs doing for those. Only if RELION or cryoSPARC wrote this "
                              "stack is the header likely right (ctf.pkl would then have a wrong -D/--Apix): set PIXEL_SIZE."))
    if box < D0:
        notes.append(("info", f"The stack is downsampled from {D0:g} to {box} px. `cryodrgn downsample` Fourier-crops "
                              f"and scales pixel values by ({D0:g}/{box})² = {(D0 / box) ** 2:.2f}; CryoSTAR's losses "
                              "ignore the overall scale, so this is fine. A full-size stack also works if you have one."))
        if int(D0) % 2:
            notes.append(("warn", f"The original box ({D0:g} px) is odd: `cryodrgn downsample` then shifts images by "
                                  "one original pixel relative to the poses."))
    consts = {}
    for col, key, label in ((5, "rlnVoltage", "voltage"), (6, "rlnSphericalAberration", "Cs"),
                            (7, "rlnAmplitudeContrast", "amplitude contrast")):
        values, freq = np.unique(np.round(ctf[:, col], 6), return_counts=True)
        consts[key] = float(values[np.argmax(freq)])
        if len(values) > 1:
            notes.append(("warn", f"The {label} varies between particles ({len(values)} values); CryoSTAR uses one, "
                                  f"so the most common, {consts[key]:g}, is written."))
    phase = np.nan_to_num(ctf[:, 8])
    if np.any(phase != 0):
        notes.append(("warn", f"{np.count_nonzero(phase):,} particles have a phase shift (up to "
                              f"{np.abs(phase).max():.0f}°). CryoSTAR's CTF ignores phase shifts, so phase-plate "
                              "data won't be modelled correctly."))

    # Image names are relative to the folder holding all stacks, so that folder's own path (which may contain
    # '@', quotes or symlinks on Google Drive) never appears in the .star; dataset_dir points at it instead.
    real = [Path(os.path.realpath(f)) for f in files]
    root = Path(os.path.commonpath([str(r.parent) for r in real]))
    rel = [str(r.relative_to(root)) for r in real]
    for name in rel:
        if any(ch in name for ch in "@#'\""):
            raise ValueError(f"The stack file name '{name}' contains '@', '#' or a quote, which a .star file can't "
                             "hold. Rename the file (and its line in the .txt list).")
    file_of_row = np.repeat(np.arange(len(files)), counts)
    index_in_file = np.concatenate([np.arange(c) for c in counts])
    names = [f"{index_in_file[r] + 1:06d}@{rel[file_of_row[r]]}" for r in stack_rows]

    origin_A = trans * box * apix  # RELION origin (x, y) in Å
    particles_df = pd.DataFrame({
        "rlnImageName": names, "rlnAngleRot": euler[:, 0], "rlnAngleTilt": euler[:, 1], "rlnAnglePsi": euler[:, 2],
        "rlnOriginXAngst": origin_A[:, 0], "rlnOriginYAngst": origin_A[:, 1],
        "rlnDefocusU": ctf[:, 2], "rlnDefocusV": ctf[:, 3], "rlnDefocusAngle": ctf[:, 4],
        "rlnPhaseShift": phase, "rlnOpticsGroup": np.ones(n, dtype=int)})
    optics_df = pd.DataFrame({
        "rlnOpticsGroupName": ["opticsGroup1"], "rlnOpticsGroup": [1], **{k: [v] for k, v in consts.items()},
        "rlnImagePixelSize": [apix], "rlnImageSize": [box], "rlnImageDimensionality": [2]})
    out_star = Path(out_star)
    out_star.parent.mkdir(parents=True, exist_ok=True)
    starfile.write({"optics": optics_df, "particles": particles_df}, out_star, overwrite=True, float_format="%.8f")

    back = starfile.read(out_star, always_dict=True)["particles"]
    angles = back[["rlnAngleRot", "rlnAngleTilt", "rlnAnglePsi"]].to_numpy(float)
    again = Rotation.from_euler("ZYZ", angles, degrees=True).as_matrix().transpose(0, 2, 1)
    if len(back) != n or (n and np.abs(again - rots).max() > 1e-5):
        raise RuntimeError("The written .star does not reproduce pose.pkl; please report this.")
    return dict(star=out_star, root=root, n=n, box=box, apix=apix, D0=D0, apix0=apix0, rows=rows_desc,
                files=files, consts=consts, notes=notes, rots=rots, origin_A=origin_A)


def find_cryodrgn_inputs(folder="", particles="", pose="", ctf="", ind=""):
    """Pick the stack and pkls, preferring the ones named in cryoDRGN's config.yaml when there is a choice.

    Returns (chosen inputs, cryoDRGN runs found, candidate ind.pkl files, .cs files in the folder).
    """
    import mrcfile, yaml
    folder = Path(folder).expanduser() if folder else None

    def resolve(given):  # a bare file name means a file in the folder
        p = Path(given).expanduser()
        return folder / p if folder is not None and not p.is_absolute() else p

    def files_in(d, *suffixes):
        return sorted(f for f in d.iterdir() if f.is_file() and f.suffix.lower() in suffixes) if d and d.is_dir() else []

    runs = []
    if folder is not None:
        if not folder.is_dir():
            raise FileNotFoundError(f"{folder} is not a folder.")
        for cfg in sorted({*folder.glob("config.yaml"), *folder.glob("*/config.yaml"), *folder.glob("*/*/config.yaml")}):
            try:
                c = yaml.safe_load(cfg.read_text()) or {}
            except Exception:
                continue
            ds = c.get("dataset_args") or {}
            if ds.get("particles"):
                runs.append(dict(config=cfg, version=c.get("version"),
                                 **{k: ds.get(k) for k in ("particles", "poses", "ctf", "ind", "invert_data")}))
    # If the folder is a training output folder, its data usually sit one level up.
    search = [folder] if folder is not None else []
    if folder is not None and (folder / "config.yaml").is_file():
        search.append(folder.parent)

    def stacks_in(d):
        found, listed = [], set()
        for t in files_in(d, ".txt"):
            lines = [l.strip() for l in t.read_text(errors="ignore").splitlines() if l.strip()]
            if lines and all(l.lower().endswith((".mrcs", ".mrc")) for l in lines) and not t.name.endswith(".ft.txt"):
                found.append(t)
                listed |= {Path(l).name for l in lines}
        for f in files_in(d, ".mrcs", ".mrc"):
            if f.name in listed or f.name.lower().endswith(".ft.mrcs"):
                continue
            try:
                with mrcfile.open(f, mode="r", header_only=True, permissive=True) as m:
                    if f.suffix.lower() == ".mrc" and int(m.header.nz) == int(m.header.nx):
                        continue  # a cubic .mrc is a map, not a particle stack
            except Exception:
                continue
            found.append(f)
        return found

    used = {k: {Path(str(r[k])).name for r in runs if r.get(k)} for k in ("particles", "poses", "ctf", "ind")}

    def pick(given, finder, key, what):
        if given:
            return resolve(given)
        if folder is None:
            raise ValueError(f"Give the {what} or CRYODRGN_FOLDER.")
        for d in search:
            candidates = finder(d)
            if len(candidates) == 1:
                return candidates[0]
            preferred = [c for c in candidates if c.name in used[key]]
            if len(preferred) == 1:
                return preferred[0]
            if candidates:
                raise ValueError(f"Several possible {what} files in {d}: " + ", ".join(c.name for c in candidates)
                                 + f". Put the file name (or full path) of the one to use in the form.")
        named = sorted(used[key])
        raise FileNotFoundError(f"No {what} found in {folder}" + (f" (cryoDRGN's config.yaml names {', '.join(named)})"
                                                                   if named else "") + "; fill in its path.")

    pkls_in = lambda d: files_in(d, ".pkl")
    chosen = dict(
        particles=pick(particles, stacks_in, "particles", "particle stack"),
        pose_pkl=pick(pose, lambda d: [p for p in pkls_in(d) if "pose" in p.name.lower()], "poses", "pose .pkl"),
        ctf_pkl=pick(ctf, lambda d: [p for p in pkls_in(d) if "ctf" in p.name.lower()], "ctf", "ctf .pkl"),
        ind_pkl=resolve(ind) if ind else None)
    ind_options = [p for d in search for p in pkls_in(d) if "ind" in p.name.lower()]
    for r in runs:  # cryoDRGN records absolute paths: they often still exist (Drive on Colab), else look in subfolders
        if r.get("ind"):
            recorded = Path(str(r["ind"]))
            found = [recorded] if recorded.is_file() else [q for d in search for q in d.glob(f"*/{recorded.name}")]
            ind_options += [q for q in found if q.is_file()]
    ind_options = list({str(p.resolve()): p for p in ind_options}.values())
    cs_files = [p for d in search for p in files_in(d, ".cs")]
    return chosen, runs, ind_options, cs_files


# ---------- the GPU check itself ----------
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True)
HAS_GPU = gpu.returncode == 0 and bool(gpu.stdout.strip())
if HAS_GPU:
    note(f"GPU: <b>{gpu.stdout.strip()}</b>", "ok")
else:
    note("No GPU found. CryoSTAR needs one: choose <b>Runtime ▸ Change runtime type ▸ T4 GPU</b>, "
         "then run this cell again.", "error")
mem_gb = int(open("/proc/meminfo").readline().split()[1]) / 1e6
print(f"CPU cores: {os.cpu_count()} · RAM: {mem_gb:.0f} GB · "
      f"free disk: {shutil.disk_usage(CONTENT).free / 1e9:.0f} GB")

In [ ]:
#@title 1.2 · (optional) Keep results on Google Drive { display-mode: "form" }
#@markdown Tick the box to write training outputs (and the tutorial download) to your Google Drive, so a disconnect never loses finished work. Colab will ask for permission to access Drive.
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_FOLDER = "CryoSTAR"  #@param {type:"string"}
#@markdown Outputs go to `MyDrive/<DRIVE_FOLDER>/work_dirs`. Writing to Drive is a bit slower than the local disk.

assert "need" in globals(), "Please run cell 1.1 first."
if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Google Drive can only be mounted inside Colab.")
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    PERSIST_DIR.mkdir(parents=True, exist_ok=True)
    note(f"Results will be saved in <code>{PERSIST_DIR}</code>.", "ok")
else:
    PERSIST_DIR = None
    note("Results stay on this Colab machine. Download them in step 5 before you close the tab.")

In [ ]:
#@title 1.3 · Install CryoSTAR { display-mode: "form" }
#@markdown Clones the code and installs the packages Colab doesn't already have (1–2 minutes). Colab's own PyTorch and NumPy are kept.
REPO_URL = "https://github.com/ts387/cryostar"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

assert "need" in globals(), "Please run cell 1.1 first."
if not (REPO_DIR / ".git").exists():
    sh(f"git clone --quiet --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
else:
    print(f"Using the existing copy in {REPO_DIR}")

# Compatibility fixes for current packages. Each one is skipped when the code already has it:
# biotite>=1.0 renamed PDBxFile to CIFFile, and recent NumPy releases removed np.row_stack.
PATCHES = {
    "cryostar/utils/pdb_tools.py": [(
        "from biotite.structure.io.pdbx import PDBxFile, get_structure\n",
        "from biotite.structure.io.pdbx import get_structure\n"
        "try:\n"
        "    from biotite.structure.io.pdbx import PDBxFile\n"
        "except ImportError:\n"
        "    # biotite>=1.0 replaced PDBxFile with CIFFile\n"
        "    from biotite.structure.io.pdbx import CIFFile as PDBxFile\n")],
    "cryostar/utils/dist_loss.py": [("np.row_stack(", "np.vstack(")],
}
for rel, edits in PATCHES.items():
    path = REPO_DIR / rel
    src = path.read_text()
    for old, new in edits:
        if old in src:
            src = src.replace(old, new)
            print(f"Applied compatibility fix to {rel}")
    path.write_text(src)

print("Installing packages…")
sh(f"{sys.executable} -m pip install -q lightning mmengine einops torchinfo umap-learn 'biotite>=1.0' "
   "biopython gemmi starfile mrcfile dm-tree tensorboard py3Dmol plotly scikit-image", quiet=True)
sh(f"{sys.executable} -m pip install -q --no-deps -e {REPO_DIR}", quiet=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))  # lets this notebook import cryostar without a restart

versions = sh(f'{sys.executable} -c "import torch, lightning, mmengine, biotite, numpy, train_atom, train_density; '
              'print(torch.__version__, torch.cuda.is_available(), lightning.__version__, '
              'mmengine.__version__, biotite.__version__, numpy.__version__)"', cwd=STAR_DIR, quiet=True)
torch_v, cuda_ok, lightning_v, mmengine_v, biotite_v, numpy_v = versions.split()[-6:]
print(f"torch {torch_v} (CUDA available: {cuda_ok}) · lightning {lightning_v} · mmengine {mmengine_v} · "
      f"biotite {biotite_v} · numpy {numpy_v}")
note("CryoSTAR is installed.", "ok")

## 2 · Data

CryoSTAR needs three things, all from a **consensus refinement** of your particles:

* **particle images** (`.mrcs` stacks),
* each particle's **pose and CTF**: either a **`.star` file** (`rlnAngleRot/Tilt/Psi`, `rlnOriginX/YAngst`, defocus) from RELION, or from cryoSPARC via `csparc2star.py`; or cryoDRGN's **`pose.pkl` + `ctf.pkl`** (from `cryodrgn parse_pose_star/csparc` and `parse_ctf_star/csparc`), which 2.1 converts to a `.star` for you,
* a **reference model** (`.pdb`/`.cif`) docked into the consensus map, in coordinates whose origin is the centre of the box (2.2 checks this and can fix it). The model and the poses must be in the same hand: if the map (or cryoDRGN's volumes) had to be hand-flipped before docking, run 2.3, which detects this and mirrors the poses to match.

The tutorial dataset already satisfies all three: 50 interpolated adenylate-kinase conformations, simulated as noisy particles with CTF.

In [ ]:
#@title 2.1 · Choose a dataset { display-mode: "form" }
DATA_SOURCE = "Tutorial: adenylate kinase (1AKE, simulated, 3.4 GB download)"  #@param ["Tutorial: adenylate kinase (1AKE, simulated, 3.4 GB download)", "My own data: a RELION or cryoSPARC .star file", "My own data: cryoDRGN files (particle stack + pose.pkl + ctf.pkl)", "My own data: upload a .zip with a .star file"]
#@markdown ---
#@markdown **Your own data.** Paths may point into Google Drive (e.g. `/content/drive/MyDrive/project/...`); Drive is mounted automatically.
REFERENCE_MODEL = ""  #@param {type:"string"}
#@markdown Atomic model (`.pdb`/`.cif`) docked into the consensus map; 2.2 checks its position.
#@markdown
#@markdown *From a `.star` file:*
STAR_FILE = ""  #@param {type:"string"}
PARTICLES_ROOT = ""  #@param {type:"string"}
#@markdown `PARTICLES_ROOT` is the folder the `rlnImageName` paths are relative to (for RELION, the project folder). Blank = detect it. After uploading a `.zip`, `STAR_FILE` and `REFERENCE_MODEL` can stay blank if it holds one of each.
#@markdown
#@markdown *From cryoDRGN files* (`pose.pkl` / `ctf.pkl` from `cryodrgn parse_pose_*` / `parse_ctf_*`, with the matching particle stack, e.g. from `cryodrgn downsample`). Give the folder and leave the rest blank to find them automatically, or give each file:
CRYODRGN_FOLDER = ""  #@param {type:"string"}
CRYODRGN_PARTICLES = ""  #@param {type:"string"}
POSE_PKL = ""  #@param {type:"string"}
CTF_PKL = ""  #@param {type:"string"}
IND_PKL = ""  #@param {type:"string"}
#@markdown `CRYODRGN_PARTICLES` is a `.mrcs` stack or the `.txt` list written by `cryodrgn downsample --chunk`; a bare file name means a file in `CRYODRGN_FOLDER`. `IND_PKL` selects particles like cryoDRGN's `--ind`; if a cryoDRGN run in the folder used `--ind` on this stack, its `ind.pkl` is used automatically.
#@markdown
#@markdown *Both:*
PIXEL_SIZE = 0.0  #@param {type:"number"}
#@markdown Å per pixel of the particle images. `0` = read it from the `.star` file or `ctf.pkl`.
COPY_TO_LOCAL_DISK = True  #@param {type:"boolean"}
#@markdown Copy the particles to Colab's disk first. Training reads images constantly, and reading them from Drive is much slower.
INVERT_CONTRAST = "auto"  #@param ["auto", "yes", "no"]
#@markdown Flip the image contrast while copying the particles. CryoSTAR expects particles as RELION and cryoSPARC write them (brighter than the background). `auto` flips a cryoDRGN stack when every cryoDRGN run on that stack used `--uninvert-data`, and nothing else. 2.3 checks the contrast against the particles either way.

need(sh="1.1 · Check the GPU and load helpers", versions="1.3 · Install CryoSTAR")
import numpy as np, mrcfile, starfile
from cryostar.utils.polymer import Polymer

TUTORIAL_URL = "https://zenodo.org/records/17581921/files/tutorial_data_1ake.zip?download=1"
TUTORIAL_PAGE = "https://zenodo.org/records/17581921"


def find_first(root, patterns):
    for pattern in patterns:
        hits = sorted(Path(root).glob(pattern))
        if hits:
            return hits[0]
    return None


def find_one(root, suffixes, what):
    hits = sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in suffixes and "__MACOSX" not in p.parts)
    if len(hits) != 1:
        listing = "".join(f"\n  {h}" for h in hits[:20]) or " none"
        raise ValueError(f"Found {len(hits)} {what} files in {root}; set the path in the form.{listing}")
    return hits[0]


def read_star(path):
    blocks = starfile.read(path, always_dict=True)
    optics = blocks.get("optics")
    particles = blocks["particles"] if "particles" in blocks else list(blocks.values())[-1]
    return optics, particles


def find_particles_root(star, image_name, user_root=""):
    rel = image_name.split("@")[-1]
    if Path(rel).is_absolute():
        if Path(rel).exists():
            return Path("/")
        raise FileNotFoundError(f"The .star file uses absolute image paths ({rel}) that don't exist here. "
                                "Make them relative to the project folder and set PARTICLES_ROOT.")
    candidates = [Path(user_root)] if user_root else [star.parent, *list(star.parent.parents)[:3]]
    for root in candidates:
        if (root / rel).exists():
            return root
    raise FileNotFoundError(f"Could not find the particle stack '{rel}' next to the .star file or in its parent "
                            "folders. Set PARTICLES_ROOT to the folder it is relative to.")


def maybe_mount_drive(*paths):
    if any(str(p).startswith("/content/drive") for p in paths) and not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")


CONVERTED = None
attention = False  # becomes True when something above needs the user's attention
invert = INVERT_CONTRAST in (True, "yes")  # "auto" can switch this on for cryoDRGN stacks below
if DATA_SOURCE.startswith("Tutorial"):
    tut_dir = DATA_ROOT / "tutorial"
    star = find_first(tut_dir, ["**/uniform_snr0-0001_ctf/simulation.star", "**/simulation.star"])
    if star is None:
        zip_path = (PERSIST_DIR or DATA_ROOT) / "tutorial_data_1ake.zip"
        if not (zip_path.exists() and zipfile.is_zipfile(zip_path)):
            print(f"Downloading the tutorial dataset from {TUTORIAL_PAGE} …")
            !wget -c -q --show-progress -O "{zip_path}.part" "{TUTORIAL_URL}"
            if not zipfile.is_zipfile(f"{zip_path}.part"):
                raise RuntimeError(f"The download failed. Download tutorial_data_1ake.zip from {TUTORIAL_PAGE} "
                                   "yourself and use the upload option or Google Drive instead.")
            os.replace(f"{zip_path}.part", zip_path)
        print("Unpacking…")
        sh(f'unzip -q -o "{zip_path}" -d "{tut_dir}"')
        if PERSIST_DIR is None:
            zip_path.unlink()  # free disk space; it is not needed after unpacking
        star = find_first(tut_dir, ["**/uniform_snr0-0001_ctf/simulation.star", "**/simulation.star"])
    pdb = find_first(tut_dir, ["**/pdbs/1akeA_50.pdb", "**/1akeA_50.pdb"])
    if star is None or pdb is None:
        raise FileNotFoundError(f"Could not find simulation.star and 1akeA_50.pdb under {tut_dir}.")
    root, base_cfg = star.parent, "1ake.py"
else:
    if "cryoDRGN" in DATA_SOURCE:
        if not REFERENCE_MODEL:
            raise ValueError("Fill in REFERENCE_MODEL in the form.")
        maybe_mount_drive(CRYODRGN_FOLDER, CRYODRGN_PARTICLES, POSE_PKL, CTF_PKL, IND_PKL, REFERENCE_MODEL)
        inputs, runs, ind_options, cs_files = find_cryodrgn_inputs(CRYODRGN_FOLDER, CRYODRGN_PARTICLES, POSE_PKL,
                                                                   CTF_PKL, IND_PKL)
        for run in runs:
            names = {k: Path(str(run[k])).name if run.get(k) else "—" for k in ("particles", "poses", "ctf", "ind")}
            print(f"Found cryoDRGN run {run['config'].parent.name} (cryoDRGN {run['version']}): particles "
                  f"{names['particles']}, poses {names['poses']}, ctf {names['ctf']}, ind {names['ind']}, "
                  f"invert_data {run['invert_data']}")
        # particles.128.txt (chunked) and particles.128.mrcs are the same particles, so match on the stem.
        same_stack = [r for r in runs if Path(str(r["particles"])).stem == inputs["particles"].stem]
        run_inds = sorted({Path(str(r["ind"])).name for r in same_stack if r.get("ind")})
        if inputs["ind_pkl"] is None and run_inds:
            hits = [p for p in ind_options if p.name in run_inds]
            if len(run_inds) == 1 and len(hits) == 1:
                inputs["ind_pkl"] = hits[0]
                users = ", ".join(r["config"].parent.name for r in same_stack if r.get("ind"))
                note(f"cryoDRGN run(s) {users} trained on this stack with <code>--ind {hits[0].name}</code>, so the "
                     "same particles are used here. Put another file in IND_PKL to change that.")
            else:
                attention = True
                note(f"cryoDRGN runs on this stack selected particles with <code>--ind {', '.join(run_inds)}</code>"
                     + ("" if hits else ", but that file isn't in the folder") + ", so ALL particles are used. "
                     "Put the ind.pkl to use in IND_PKL.", "warn")
        uninverted = [r["config"].parent.name for r in same_stack if r.get("invert_data") is False]
        default = [r["config"].parent.name for r in same_stack if r.get("invert_data") is True]
        if INVERT_CONTRAST == "auto" and uninverted and not default:
            invert = True
            note(f"cryoDRGN run(s) {', '.join(uninverted)} trained on this stack with <code>--uninvert-data</code>, so "
                 "its contrast is the opposite of what CryoSTAR expects and is flipped while copying "
                 "(INVERT_CONTRAST = auto). 2.3 checks this against the particles.")
        elif uninverted and not invert and INVERT_CONTRAST == "auto":
            attention = True
            note(f"cryoDRGN run(s) {', '.join(uninverted)} trained on this stack with <code>--uninvert-data</code> "
                 f"but {', '.join(default)} did not, so the contrast is left as it is. 2.3 checks it against the "
                 "particles; set INVERT_CONTRAST to yes or no to decide yourself.", "warn")
        elif uninverted and not invert:
            note(f"cryoDRGN run(s) {', '.join(uninverted)} trained on this stack with <code>--uninvert-data</code>; "
                 "the contrast is left as it is because INVERT_CONTRAST is no. 2.3 checks it against the particles.")
        others = [r["config"].parent.name for r in runs if r not in same_stack and r.get("invert_data") is False]
        if not same_stack and others and not invert:
            attention = True
            note(f"cryoDRGN run(s) {', '.join(others)} used <code>--uninvert-data</code> on another stack in this "
                 "folder. If that stack holds the same particles, this one probably needs INVERT_CONTRAST = yes "
                 "too; 2.3 checks it against the particles.", "warn")
        print("Using " + ", ".join(f"{k} {v}" for k, v in inputs.items() if v))

        def converted_star(inputs):
            """One .star per selection (stack, pkls and ind), so different selections never share a file name."""
            import hashlib
            sig = "|".join(str(Path(str(inputs[k])).resolve()) if inputs.get(k) else "" for k in sorted(inputs))
            tag = hashlib.sha1(sig.encode()).hexdigest()[:6]
            return DATA_ROOT / "cryodrgn" / f"{inputs['particles'].stem}_{tag}_cryostar.star"

        out_star = converted_star(inputs)
        try:
            CONVERTED = cryodrgn_to_star(**inputs, out_star=out_star, apix=PIXEL_SIZE or None)
        except RowCountError:
            if inputs["ind_pkl"] is not None or len(ind_options) != 1:
                raise
            note(f"The stack and pkls have different lengths, so <code>{ind_options[0].name}</code> is used to "
                 "match them. Give IND_PKL explicitly if that is the wrong file.", "warn")
            inputs["ind_pkl"] = ind_options[0]
            out_star = converted_star(inputs)
            CONVERTED = cryodrgn_to_star(**inputs, out_star=out_star, apix=PIXEL_SIZE or None)
        star, root, pdb = CONVERTED["star"], CONVERTED["root"], Path(REFERENCE_MODEL).expanduser()
        for kind, msg in CONVERTED["notes"]:
            attention |= kind == "warn"
            note(msg, kind)
        # cryoDRGN's parse_pose_csparc takes cryoSPARC shifts to be in pixels of the particle images. If
        # cryoSPARC aligned at another pixel size, every shift in pose.pkl is off by that ratio.
        for cs in cs_files:
            try:
                meta = np.load(cs, mmap_mode="r")
                ratio = np.asarray(meta["alignments3D/psize_A"], float) / np.asarray(meta["blob/psize_A"], float)
            except Exception:
                continue
            if np.nanmax(np.abs(ratio - 1)) > 1e-3:
                attention = True
                note(f"In <code>{cs.name}</code> cryoSPARC aligned the particles at {np.nanmedian(ratio):.3g}× the "
                     "pixel size of the stored images. cryoDRGN's parse_pose_csparc ignores this, so if pose.pkl "
                     "came from this file its shifts are scaled wrongly. Re-run the refinement on the final "
                     "particles in cryoSPARC, then parse the new .cs.", "warn")
        print(f"Converted to {star}: {CONVERTED['n']:,} particles ({CONVERTED['rows']}), {CONVERTED['box']} px "
              f"at {CONVERTED['apix']:.4f} Å/px (original {CONVERTED['D0']:g} px at {CONVERTED['apix0']:g} Å/px)")
    elif "upload" in DATA_SOURCE:
        if not IN_COLAB:
            raise RuntimeError("Uploading only works inside Colab; use the path option instead.")
        from google.colab import files
        up_dir = DATA_ROOT / "upload"
        up_dir.mkdir(exist_ok=True)
        print("Choose a .zip with your .star file, particle stacks and reference model…")
        for name in files.upload():
            shutil.move(name, up_dir / name)
            if name.lower().endswith(".zip"):
                sh(f'unzip -q -o "{up_dir / name}" -d "{up_dir}"')
                (up_dir / name).unlink()
        star = Path(STAR_FILE) if STAR_FILE else find_one(up_dir, {".star"}, ".star")
        pdb = Path(REFERENCE_MODEL) if REFERENCE_MODEL else find_one(up_dir, {".pdb", ".cif"}, "model (.pdb/.cif)")
    else:
        if not STAR_FILE or not REFERENCE_MODEL:
            raise ValueError("Fill in STAR_FILE and REFERENCE_MODEL in the form.")
        maybe_mount_drive(STAR_FILE, REFERENCE_MODEL, PARTICLES_ROOT)
        star, pdb = Path(STAR_FILE).expanduser(), Path(REFERENCE_MODEL).expanduser()
    for p in (star, pdb):
        if not p.is_file():
            raise FileNotFoundError(f"{p} does not exist.")
    _, particles = read_star(star)
    if CONVERTED is None:
        root = find_particles_root(star, particles["rlnImageName"].iloc[0], PARTICLES_ROOT)
    base_cfg = "example.py"

    stacks = sorted(set(particles["rlnImageName"].str.split("@").str[-1]))
    absolute = Path(stacks[0]).is_absolute()
    inside = lambda path, folder: str(Path(path).resolve()).startswith(str(Path(folder).resolve()))
    if invert and absolute:
        raise RuntimeError("INVERT_CONTRAST needs a .star file with relative image paths (see PARTICLES_ROOT).")
    if invert and inside(root, DATA_ROOT / "own"):
        raise RuntimeError("These particles are already a copy made by this notebook; give the original .star file "
                           "to invert the contrast.")
    if not absolute and not inside(root, DATA_ROOT / "own") and (invert or (COPY_TO_LOCAL_DISK and
                                                                            not inside(root, DATA_ROOT))):
        import hashlib
        # One cache folder per source folder and contrast, refreshed whenever a source file changes. Only the
        # particle stacks are copied; the .star file and the model are used where they are.
        source = str(Path(root).resolve())
        key = hashlib.sha1(f"{source}|{invert}".encode()).hexdigest()[:10]
        local = DATA_ROOT / "own" / key
        manifest_path = local / ".sources.json"
        manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
        manifest.update(__source__=source, __invert__=invert)
        signature = {s: [(root / s).stat().st_size, int((root / s).stat().st_mtime)] for s in stacks}
        todo = [s for s in stacks if manifest.get(s) != signature[s] or not (local / s).exists()]

        def copy_size(s):
            if not invert:
                return signature[s][0]
            with mrcfile.mmap(root / s, mode="r", permissive=True) as m:  # written as float32 (float16 kept)
                return m.data.size * (2 if m.data.dtype == np.float16 else 4) + 1024

        size = sum(copy_size(s) for s in todo)
        free = shutil.disk_usage(CONTENT).free
        gb = lambda b: f"{b / 1e9:,.1f} GB" if b > 1e8 else f"{b / 1e6:,.0f} MB"
        if size > free - 5e9:
            # Make room by removing local copies of other particle stacks (they are copied again when needed).
            in_use = {(local / s).resolve() for s in stacks}
            spare = [f for f in (DATA_ROOT / "own").rglob("*") if f.is_file() and f.name != ".sources.json"
                     and f.resolve() not in in_use]
            if spare and size <= free + sum(f.stat().st_size for f in spare) - 5e9:
                print(f"Removing {len(spare)} local copies of other particle stacks "
                      f"({gb(sum(f.stat().st_size for f in spare))}) to make room:")
                for f in spare:
                    print(f"  {f}")
                    f.unlink()
                free = shutil.disk_usage(CONTENT).free
        if size > free - 5e9:
            if invert:
                raise RuntimeError(f"Inverting the contrast needs a local copy of {gb(size)} (plus 5 GB to spare), "
                                   f"but only {gb(free)} is free. Free some space on the Colab disk, or use a "
                                   "smaller stack.")
            note(f"The particles take {gb(size)} but only {gb(free)} is free (5 GB is kept spare), "
                 "so they will be read from where they are.", "warn")
        else:
            local.mkdir(parents=True, exist_ok=True)
            if todo:
                from tqdm.auto import tqdm
                print(f"{'Inverting and copying' if invert else 'Copying'} {len(todo)} particle stack(s), "
                      f"{gb(size)}, to the local disk…")
                bar = tqdm(total=size, unit="B", unit_scale=True, unit_divisor=1024)
                for s in todo:
                    src, dst = root / s, local / s
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    if invert:
                        with mrcfile.mmap(src, mode="r", permissive=True) as m_in:
                            mode = 12 if m_in.data.dtype == np.float16 else 2
                            with mrcfile.new_mmap(dst, shape=m_in.data.shape, mrc_mode=mode, overwrite=True) as m_out:
                                for i in range(0, len(m_in.data), 2000):
                                    block = -np.asarray(m_in.data[i:i + 2000], dtype=m_out.data.dtype)
                                    m_out.data[i:i + 2000] = block
                                    bar.update(block.nbytes)
                                m_out.voxel_size = m_in.voxel_size
                    else:
                        with open(src, "rb") as f_in, open(dst, "wb") as f_out:
                            while chunk := f_in.read(64 << 20):
                                f_out.write(chunk)
                                bar.update(len(chunk))
                    manifest[s] = signature[s]
                    manifest_path.write_text(json.dumps(manifest))
                bar.close()
            else:
                print(f"Using the local copy of the particles in {local}")
            manifest_path.write_text(json.dumps(manifest))
            root = local
    if invert and not inside(root, DATA_ROOT / "own"):
        raise RuntimeError("The contrast could not be inverted because the particles were not copied (see above).")

# Read the metadata CryoSTAR needs and check it before any training starts.
optics, particles = read_star(star)
required = ["rlnImageName", "rlnDefocusU", "rlnDefocusV", "rlnDefocusAngle",
            "rlnAngleRot", "rlnAngleTilt", "rlnAnglePsi"]
missing = [c for c in required if c not in particles]
if not ({"rlnOriginXAngst", "rlnOriginYAngst"} <= set(particles) or {"rlnOriginX", "rlnOriginY"} <= set(particles)):
    missing.append("rlnOriginXAngst/rlnOriginYAngst")
for c in ("rlnVoltage", "rlnSphericalAberration", "rlnAmplitudeContrast"):
    if not ((optics is not None and c in optics) or c in particles):
        missing.append(c)
if missing:
    raise ValueError("The .star file is missing columns CryoSTAR needs: " + ", ".join(missing))
if optics is not None and len(optics) > 1:
    raise ValueError(f"The .star file has {len(optics)} optics groups; CryoSTAR supports one. "
                     "Split the particles per optics group (or merge them) first.")

if PIXEL_SIZE > 0:
    apix = float(PIXEL_SIZE)
elif optics is not None and "rlnImagePixelSize" in optics:
    apix = float(optics["rlnImagePixelSize"].iloc[0])
elif {"rlnDetectorPixelSize", "rlnMagnification"} <= set(particles):
    apix = float(particles["rlnDetectorPixelSize"].iloc[0] / particles["rlnMagnification"].iloc[0] * 1e4)
elif base_cfg == "1ake.py":
    apix = 1.0  # set in the tutorial config
else:
    raise ValueError("Could not read the pixel size from the .star file; enter it as PIXEL_SIZE.")
first_stack = root / particles["rlnImageName"].iloc[0].split("@")[-1]
with mrcfile.mmap(first_stack, mode="r", permissive=True) as m:
    box = int(m.data.shape[-1])
model = Polymer.from_pdb(str(pdb))

if CONVERTED is not None:
    # Read a sample of particles back through CryoSTAR's own loader and compare with pose.pkl directly.
    from cryostar.utils.dataio import StarfileDataSet, StarfileDatasetConfig
    import contextlib, io
    with contextlib.redirect_stdout(io.StringIO()):
        dataset = StarfileDataSet(StarfileDatasetConfig(starfile_path=str(star), dataset_dir=str(root),
                                                        apix=apix, side_shape=box))
    sample = np.unique(np.linspace(0, len(dataset) - 1, min(len(dataset), 200)).astype(int))
    rot_err = shift_err = 0.0
    empty = 0
    for i in sample:
        with contextlib.redirect_stdout(io.StringIO()):
            item = dataset[int(i)]
        empty += int(float(item["proj"].abs().max()) == 0)
        rot_err = max(rot_err, float(np.abs(item["rotmat"].numpy() - CONVERTED["rots"][i]).max()))
        shift = np.array([float(item["shiftX"]), float(item["shiftY"])])
        shift_err = max(shift_err, float(np.abs(shift - CONVERTED["origin_A"][i]).max()))
    if empty or rot_err > 1e-4 or shift_err > 1e-3:
        raise RuntimeError(f"CryoSTAR does not read the converted particles as expected ({empty} unreadable "
                           f"images, rotation difference {rot_err:.2e}, shift difference {shift_err:.2e} Å).")
    note(f"Checked {len(sample)} particles through CryoSTAR's own loader: rotations match pose.pkl to "
         f"{rot_err:.1e} and shifts to {shift_err:.1e} Å, and every image reads.", "ok")

DATA = dict(kind="tutorial" if base_cfg == "1ake.py" else "own", star=star, root=root, pdb=pdb,
            apix=apix, box=box, n=len(particles), base=base_cfg,
            n_aa=int(model.num_amino_acids), n_nt=int(model.num_nucleotides),
            apix_override=PIXEL_SIZE > 0, inverted_contrast=invert and base_cfg != "1ake.py")
display(HTML(f"""<table style="text-align:left">
<tr><th>Particles</th><td>{DATA['n']:,} images of {box}×{box} px at {apix:.3f} Å/px</td></tr>
<tr><th>Reference model</th><td>{DATA['n_aa']:,} amino acids, {DATA['n_nt']:,} nucleotides, {model.num_chains} chain(s)</td></tr>
<tr><th>.star file</th><td><code>{star}</code></td></tr>
<tr><th>Particles root</th><td><code>{root}</code></td></tr>
<tr><th>Model file</th><td><code>{pdb}</code></td></tr></table>"""))
if attention:
    note("Dataset ready, but please read the warnings above before training.", "warn")
else:
    note("Dataset ready.", "ok")

In [ ]:
#@title 2.2 · (recommended) Look at the data and check the model's position { display-mode: "form" }
#@markdown Shows a few particles and the reference model, and checks that the model is centred in the box the way CryoSTAR expects.
#@markdown
#@markdown **Own data:** give a consensus map reconstructed from these particles' poses (e.g. the cryoSPARC/RELION refinement map or cryoDRGN's consensus volume). If you docked the model in ChimeraX on a map whose origin is at its corner, tick `CENTER_MODEL` to move the model's origin to the box centre (this runs CryoSTAR's `cstar_center_origin`).
#@markdown
#@markdown The model is then fitted to the map and to its mirror images. If it only fits a mirror image (the map had to be hand-flipped before docking), the poses are in the other hand: 2.3 will mirror them, using a copy of the model moved by the small offset that the fit measures, so that it is the exact mirror of the structure the poses describe. The map doesn't need to be flipped first.
CONSENSUS_MAP = ""  #@param {type:"string"}
CENTER_MODEL = False  #@param {type:"boolean"}

need(DATA="2.1 · Choose a dataset")
import numpy as np, mrcfile, torch, matplotlib.pyplot as plt, py3Dmol
from cryostar.utils.polymer import Polymer
from cryostar.utils.pdb_tools import bt_read_pdb, bt_save_pdb
from cryostar.gmm.gmm import EMAN2Grid, Gaussian, canonical_density

MIRROR = {"x": np.diag([-1.0, 1, 1]), "y": np.diag([1.0, -1, 1]), "z": np.diag([1.0, 1, -1])}


def model_density(coords, electrons, box, voxel):
    """CryoSTAR's Gaussian-bead density of a model (coordinates centred on the box), as a (z, y, x) array."""
    dens = canonical_density(Gaussian(mus=torch.tensor(coords, dtype=torch.float32),
                                      sigmas=torch.full((len(coords),), 2.0),
                                      amplitudes=torch.tensor(electrons, dtype=torch.float32)),
                             EMAN2Grid(side_shape=box, voxel_size=voxel).line())
    return dens.permute(2, 1, 0).numpy().astype(np.float64)


def fourier_crop(vol, voxel, max_box=128):
    """Shrink a cubic map to at most max_box voxels by Fourier cropping; the physical box stays the same."""
    D = vol.shape[0]
    vol = vol.astype(np.float64)
    if D <= max_box:
        return vol, voxel
    f = np.fft.fftshift(np.fft.fftn(vol))
    c = (D - max_box) // 2
    return np.fft.ifftn(np.fft.ifftshift(f[c:c + max_box, c:c + max_box, c:c + max_box])).real, voxel * D / max_box


def register_model(map_data, voxel, coords, electrons, res=8.0, max_box=128):
    """Fit the model to the map as is and mirrored along x/y/z, with a sub-voxel translation search.

    Returns {variant: (correlation, translation (x, y, z) in Å, mirror matrix)} with map ≈ density(M·coords + t).
    """
    vol, voxel = fourier_crop(map_data, voxel, max_box)
    D = vol.shape[0]
    k = np.fft.fftfreq(D, voxel)
    kz, ky, kx = np.meshgrid(k, k, k, indexing="ij")
    lowpass = np.sqrt(kx ** 2 + ky ** 2 + kz ** 2) < 1 / max(res, 2.5 * voxel)
    F_map = np.fft.fftn(vol - vol.mean()) * lowpass
    fits = {}
    for name, M in [("as is", np.eye(3))] + [(f"mirrored along {a}", m) for a, m in MIRROR.items()]:
        dens = model_density(coords @ M.T, electrons, D, voxel)
        F_mod = np.fft.fftn(dens - dens.mean()) * lowpass
        cc = np.fft.ifftn(F_map * np.conj(F_mod)).real
        i = np.array(np.unravel_index(np.argmax(cc), cc.shape))
        sub = []
        for ax in range(3):
            e = np.eye(3, dtype=int)[ax]
            c0, cm, cp = cc[tuple(i % D)], cc[tuple((i - e) % D)], cc[tuple((i + e) % D)]
            sub.append(0.5 * (cm - cp) / (cm - 2 * c0 + cp) if cm - 2 * c0 + cp != 0 else 0.0)
        t_zyx = ((i + np.array(sub) + D / 2) % D - D / 2) * voxel
        norm = np.sqrt((np.abs(F_map) ** 2).sum() * (np.abs(F_mod) ** 2).sum())
        fits[name] = (float(cc.max() * D ** 3 / norm), t_zyx[::-1], M)
    return fits


# Start from the model chosen in 2.1 and the poses as given (an earlier 2.2 may have centred the model, 2.3 may have
# switched to a mirror-fitted copy and mirrored poses) and forget earlier fits; 2.3 has to be run again afterwards.
DATA["pdb"] = DATA.setdefault("pdb_original", DATA.get("pdb_as_docked", DATA["pdb"]))
DATA["star"] = DATA.get("star_as_given", DATA["star"])
reset_23 = "pose_check" in DATA
for key in ("hand_hint", "pdb_mirror_fit", "pdb_as_docked", "star_as_given", "hand_flip", "pose_check"):
    DATA.pop(key, None)
if reset_23:
    note("2.2 starts again from your model and the poses as given, so run 2.3 again afterwards.")

_, particles = read_star(DATA["star"])
# A few particles, evenly spread through the dataset.
picks = np.linspace(0, DATA["n"] - 1, 8).astype(int)
fig, axes = plt.subplots(1, len(picks), figsize=(2 * len(picks), 2.3))
for ax, i in zip(axes, picks):
    idx, name = particles["rlnImageName"].iloc[i].split("@")
    with mrcfile.mmap(DATA["root"] / name, mode="r", permissive=True) as m:
        img = m.data[int(idx) - 1] if m.data.ndim == 3 else m.data
        ax.imshow(np.asarray(img), cmap="gray")
    ax.set_title(f"#{i}", fontsize=9)
    ax.axis("off")
fig.suptitle("Particle images (individual particles are noisy; that is expected)", fontsize=10)
plt.tight_layout()
plt.show()

if CONSENSUS_MAP:
    maybe_mount_drive(CONSENSUS_MAP)
    prep = DATA_ROOT / "prep"
    prep.mkdir(exist_ok=True)
    with mrcfile.open(CONSENSUS_MAP, permissive=True) as m:
        cmap_data, vs, origin = m.data.copy(), float(m.voxel_size.x), m.header.origin
    origin = np.array([origin.x, origin.y, origin.z], dtype=float)
    print(f"Consensus map: {cmap_data.shape} voxels, {vs:.3f} Å/voxel, origin {origin} Å")
    if cmap_data.ndim != 3 or len(set(cmap_data.shape)) != 1:
        raise ValueError(f"The consensus map must be a cube; this one is {cmap_data.shape} voxels.")
    map_box, particle_box = cmap_data.shape[0] * vs, DATA["box"] * DATA["apix"]
    if abs(map_box / particle_box - 1) > 0.02:
        note(f"The map's box is {map_box:.1f} Å across but the particles' is {particle_box:.1f} Å. Check the map's "
             "pixel size (its header) and that it was reconstructed from these particles.", "warn")
    if CENTER_MODEL and Path(DATA["pdb"]).stem.endswith("_centered"):
        note("The selected model is already the centred copy, so it was not shifted again.")
    elif CENTER_MODEL:
        if np.abs(origin).max() > 1e-3:
            note("This map's origin is not at its corner, so the model may already be centred. "
                 "Check the overlay below before training.", "warn")
        sh(f'cstar_center_origin "{DATA["pdb"]}" "{CONSENSUS_MAP}"', cwd=prep)
        DATA["pdb"] = prep / (Path(DATA["pdb"]).stem + "_centered.pdb")
        note(f"Centred model written to <code>{DATA['pdb']}</code> and selected for training.", "ok")
    fitted = Polymer.from_pdb(str(DATA["pdb"]))
    fits = register_model(cmap_data, vs, fitted.coord, fitted.num_electron)
    best = max(fits, key=lambda n: fits[n][0])
    display(HTML("<p>Fit of the model to this map (correlation at 8 Å after the best translation):</p><table>"
                 "<tr><th>model</th><th>correlation</th><th>translation needed (Å)</th></tr>" + "".join(
                     f"<tr><td>{'<b>' if n == best else ''}{n}{'</b>' if n == best else ''}</td><td>{c:.3f}</td>"
                     f"<td>({t[0]:+.1f}, {t[1]:+.1f}, {t[2]:+.1f})</td></tr>" for n, (c, t, _) in fits.items())
                 + "</table>"))
    corr_as_is, t_as_is, _ = fits["as is"]
    shown, shown_label = fitted.coord, "the model"
    if best != "as is" and fits[best][0] > corr_as_is + 0.05:
        corr, t, M = fits[best]
        axis = best[-1]
        if np.abs(t).max() > 3 * vs:
            note(f"The model fits this map best {best}, but only after moving it by {np.abs(t).max():.0f} Å, so it "
                 "isn't positioned in this map's box. Centre it (CENTER_MODEL) or check the docking first.", "warn")
        else:
            shift = M @ t  # puts the model where it is the exact mirror (about the box centre) of the map's structure
            atoms = bt_read_pdb(str(DATA["pdb"]))[0]
            atoms.coord = atoms.coord + shift.astype(atoms.coord.dtype)
            out = prep / f"{Path(DATA['pdb']).stem}_mirrorfit_{axis}{Path(DATA['pdb']).suffix}"
            bt_save_pdb(str(out), atoms)
            DATA["hand_hint"], DATA["pdb_mirror_fit"] = axis, out
            shown, shown_label = fitted.coord @ M.T + t, f"the model mirrored along {axis}"
            note(f"The model fits this map's <b>mirror image</b> (mirrored along {axis}; correlation {corr:.2f} vs "
                 f"{corr_as_is:.2f} as is). If this map was reconstructed from these particles' poses, the poses are "
                 f"in the other hand from your model: 2.3 will mirror them and then use a copy of the model moved by "
                 f"({shift[0]:+.2f}, {shift[1]:+.2f}, {shift[2]:+.2f}) Å, which makes it the exact mirror of the "
                 "structure in this map.", "warn")
    elif np.abs(t_as_is).max() > 1.5 * vs:
        note(f"The model fits this map best after moving it by ({t_as_is[0]:+.1f}, {t_as_is[1]:+.1f}, "
             f"{t_as_is[2]:+.1f}) Å. Check the docking (or the centring) before training.", "warn")
    elif corr_as_is >= 0.3:
        note(f"The model fits this map as it is (correlation {corr_as_is:.2f}).", "ok")
    if fits[best][0] < 0.3:
        note(f"Even the best fit is poor (correlation {fits[best][0]:.2f}). Check that this map was reconstructed "
             "from these particles, that its pixel size is right and that the model is docked in it.", "warn")
    shown_map, shown_vs = fourier_crop(cmap_data, vs)  # at most 128 voxels, to keep the overlay light
    overlay = model_density(shown, fitted.num_electron, shown_map.shape[0], shown_vs)
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
    for ax, axis_idx, label in zip(axes, (0, 1, 2), ("down z", "down y", "down x")):
        ax.imshow(shown_map.sum(axis_idx), cmap="gray")
        proj = overlay.sum(axis_idx)
        ax.contour(proj, levels=[proj.max() * 0.15], colors=[SERIES_ORANGE], linewidths=1.5)
        ax.set_title(f"Projection {label}", fontsize=10)
        ax.axis("off")
    fig.suptitle(f"Consensus map (grey) with the outline of {shown_label} (orange): they should overlap", fontsize=10)
    plt.tight_layout()
    plt.show()
DATA["pdb_as_docked"] = DATA["pdb"]

# CryoSTAR places the model so that (0, 0, 0) is the centre of the box.
model = Polymer.from_pdb(str(DATA["pdb"]))
centre = model.coord.mean(0)
half_box = DATA["box"] * DATA["apix"] / 2
radius = np.linalg.norm(model.coord - centre, axis=1).max()
print(f"Model centre: ({centre[0]:.1f}, {centre[1]:.1f}, {centre[2]:.1f}) Å; box half-width: {half_box:.1f} Å")
if np.abs(centre).max() > 0.25 * half_box:
    note("The model's centre is far from (0, 0, 0). If it was docked with the map origin at the corner, "
         "give CONSENSUS_MAP, tick CENTER_MODEL and run this cell again.", "warn")
elif np.abs(centre).max() + radius > 1.05 * half_box:
    note("Parts of the model stick out of the box. Check that the pixel size and the docking are right.", "warn")
elif np.abs(centre).max() > 10:
    note(f"The model sits inside the box, but its centre is {np.linalg.norm(centre):.0f} Å from the box centre. "
         "That's fine if the particle is off-centre in the box; otherwise check the docking.")
else:
    note("The model sits inside the box, centred as CryoSTAR expects.", "ok")

view = py3Dmol.view(width=640, height=380)
view.addModel(Path(DATA["pdb"]).read_text(), "pdb" if Path(DATA["pdb"]).suffix == ".pdb" else "cif")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

In [ ]:
#@title 2.3 · (recommended) Check the poses against the model: hand, contrast, docking { display-mode: "form" }
#@markdown CryoSTAR projects your reference model with every particle's pose, so the poses and the model must belong to the same hand. If the map had to be **hand-flipped before you docked the model** (common when the consensus refinement settled on the mirror image), the poses are in the other hand. **Run this cell whenever the map, or cryoDRGN's volumes, were ever hand-flipped.**
#@markdown
#@markdown It projects the model (with each particle's CTF and shift) using the poses as they are, mirrored for a map flipped along x, y or z (R → M<sub>z</sub>·R·M<sub>k</sub>; shifts unchanged), and shuffled between particles as a baseline, and correlates each with the particle images. `auto-detect` mirrors the poses only when that fits clearly better, the poses clearly matter (they beat the shuffled baseline) and the model is docked (turning it by 30° doesn't fit better); when the particles can't tell the hands apart and don't contradict it, it follows 2.2's map fit. It also detects inverted contrast. cryoDRGN `flip_hand`, ChimeraX `volume flip` and cryoSPARC flip along **z**; `relion_image_handler --invert_hand` flips along **x**.
#@markdown
#@markdown Tools that reverse the map array (e.g. cryoDRGN `flip_hand`) mirror it about voxel (N−1)/2, whereas mirrored poses mirror the structure about the rotation centre N/2 that cryoDRGN and CryoSTAR share. A model docked in such a flipped map is therefore one voxel from where the mirrored poses put the structure; that is not a docking error. Give 2.2 the map in the poses' hand (e.g. cryoDRGN's unflipped consensus volume) and it writes a copy of the model moved by that voxel, so CryoSTAR's results sit one voxel from your docked model along the flip axis.
HAND = "auto-detect"  #@param ["auto-detect", "keep the poses as they are", "flip (map was flipped along z)", "flip (map was flipped along x)", "flip (map was flipped along y)"]
CHECK_PARTICLES = 1000  #@param {type:"integer"}
CHECK_RESOLUTION = 15.0  #@param {type:"number"}
#@markdown Images and projections are compared after low-pass filtering to `CHECK_RESOLUTION` Å (at most 96 px are used).

need(DATA="2.1 · Choose a dataset")
import contextlib, hashlib, io, warnings
import numpy as np, torch, starfile, matplotlib.pyplot as plt
from torch.utils.data import default_collate
from scipy.spatial.transform import Rotation
from cryostar.utils.dataio import StarfileDataSet, StarfileDatasetConfig
from cryostar.utils.polymer import Polymer
from cryostar.utils.ctf import parse_ctf_star
from cryostar.utils.ctf_utils import CTFCryoDRGN
from cryostar.utils.fft_utils import primal_to_fourier_2d, fourier_to_primal_2d
from cryostar.utils.transforms import SpatialGridTranslate
from cryostar.gmm.gmm import EMAN2Grid, Gaussian, batch_projection

MIRROR = {"x": np.diag([-1.0, 1, 1]), "y": np.diag([1.0, -1, 1]), "z": np.diag([1.0, 1, -1])}
DATA.pop("pose_check", None)
star_in = DATA.setdefault("star_as_given", DATA["star"])  # always start from the poses as given in 2.1
pdb_as_docked = Path(DATA.setdefault("pdb_as_docked", DATA["pdb"]))
hint, pdb_fit = DATA.get("hand_hint"), DATA.get("pdb_mirror_fit")  # from the map fit in 2.2
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- particles: a small box is plenty at this resolution
D = min(DATA["box"], 96) // 2 * 2
apix_d = DATA["box"] * DATA["apix"] / D
res = max(float(CHECK_RESOLUTION), 2.5 * apix_d)
with contextlib.redirect_stdout(io.StringIO()):
    dataset = StarfileDataSet(StarfileDatasetConfig(starfile_path=str(star_in), dataset_dir=str(DATA["root"]),
                                                    apix=DATA["apix"], side_shape=DATA["box"], down_side_shape=D,
                                                    down_method="fft"))
    picks = np.unique(np.linspace(0, len(dataset) - 1, min(len(dataset), max(CHECK_PARTICLES, 50))).astype(int))
    batch = default_collate([dataset[int(i)] for i in picks])
finite = torch.isfinite(batch["proj"].flatten(1)).all(1)
batch["proj"] = torch.nan_to_num(batch["proj"], nan=0.0, posinf=0.0, neginf=0.0)
blank = (~finite | (batch["proj"].flatten(1).abs().amax(1) == 0)).numpy()
if blank.all():
    raise RuntimeError("None of the sampled particle images could be read. Check the particle files chosen in 2.1 "
                       "(PARTICLES_ROOT, and that every stack named in the .star exists).")

freq = np.fft.fftshift(np.fft.fftfreq(D, apix_d))
lowpass = torch.tensor(np.sqrt(freq[:, None] ** 2 + freq[None, :] ** 2) < 1 / res, dtype=torch.float32, device=device)
yy, xx = np.mgrid[:D, :D] - D // 2
mask = torch.tensor(np.hypot(yy, xx) < 0.45 * D, device=device)
filt = lambda img: fourier_to_primal_2d(primal_to_fourier_2d(img) * lowpass).real
with torch.no_grad():
    trans = torch.stack([batch["shiftY"], batch["shiftX"]], dim=1).float().reshape(-1, 1, 2) / apix_d
    images = SpatialGridTranslate(D=D, device=device).transform(batch["proj"][:, 0].to(device), trans.to(device))
    images = filt(images[:, 0])

model = Polymer.from_pdb(str(pdb_as_docked))
mus = torch.tensor(model.coord, dtype=torch.float32, device=device)[None]
mus_fit = (torch.tensor(Polymer.from_pdb(str(pdb_fit)).coord, dtype=torch.float32, device=device)[None]
           if pdb_fit else mus)
amps = torch.tensor(model.num_electron, dtype=torch.float32, device=device)[None]
sigmas = torch.full_like(amps, 2.0)
grid = EMAN2Grid(side_shape=D, voxel_size=apix_d).to(device).line()
ctf_row = parse_ctf_star(star_in, side_shape=D, apix=apix_d)[0]
ctf = CTFCryoDRGN(size=D, resolution=apix_d, kV=float(ctf_row[5]), cs=float(ctf_row[6]),
                  amplitudeContrast=float(ctf_row[7])).to(device)
ctf_params = {k: batch[k].to(device) for k in ("defocusU", "defocusV", "angleAstigmatism")}
step = max(1, min(64, int(2e8 // (D * mus.shape[1]))))  # bound memory for large models


def masked_corr(a, b):
    a = a[:, mask] - a[:, mask].mean(1, keepdim=True)
    b = b[:, mask] - b[:, mask].mean(1, keepdim=True)
    return (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + 1e-12)


def project(rots, coords):
    out = []
    for s in range(0, len(rots), step):
        r = torch.tensor(rots[s:s + step], dtype=torch.float32, device=device)
        b = len(r)
        p = batch_projection(Gaussian(mus=coords.expand(b, -1, -1), sigmas=sigmas.expand(b, -1),
                                      amplitudes=amps.expand(b, -1)), r, grid)
        f = ctf(primal_to_fourier_2d(p[:, None]), None, {k: v[s:s + step] for k, v in ctf_params.items()})
        out.append(filt(fourier_to_primal_2d(f).real[:, 0]))
    return torch.cat(out)


def score(rots, coords):
    with torch.no_grad():
        p = project(rots, coords)
        return masked_corr(images, p).double().cpu().numpy(), p.cpu()


rot = batch["rotmat"].double().numpy()
candidates = {"as given": (rot, mus)}
for k, m in MIRROR.items():
    candidates[f"flipped along {k}"] = (MIRROR["z"] @ rot @ m, mus_fit if k == hint else mus)
candidates["shuffled poses"] = (rot[np.random.default_rng(0).permutation(len(rot))], mus)
scores, projections = {}, {}
for name, (r, c) in candidates.items():
    scores[name], projections[name] = score(r, c)

# Leave out unreadable (blank) images and any particle with a non-finite score.
keep = ~blank & np.isfinite(np.stack(list(scores.values()))).all(0)
if not keep.all():
    note(f"{(~keep).sum()} of {len(keep)} sampled particles were left out of the check (unreadable images or "
         "invalid values). If that is more than a few, check the particle files chosen in 2.1.", "warn")
scores = {k: v[keep] for k, v in scores.items()}
n = int(keep.sum())
paired = lambda a, b: float((scores[a] - scores[b]).mean() / ((scores[a] - scores[b]).std(ddof=1) / np.sqrt(n) + 1e-12))
hands = [k for k in scores if k != "shuffled poses"]
flip_names = [k for k in hands if k != "as given"]

# Contrast: with inverted contrast the poses that fit best give the most negative correlation.
strongest = max(hands, key=lambda k: abs(scores[k].mean()))
inverted = n >= 10 and scores[strongest].mean() < 0 and paired("shuffled poses", strongest) > 5
if inverted:
    scores = {k: -v for k, v in scores.items()}
best = max(hands, key=lambda k: scores[k].mean())
best_flip = max(flip_names, key=lambda k: scores[k].mean())
second_flip = sorted(flip_names, key=lambda k: -scores[k].mean())[1]
pose_z = paired(best, "shuffled poses") if n >= 10 else float("nan")
pose_frac = (scores[best].mean() - scores["shuffled poses"].mean()) / (abs(scores[best].mean()) + 1e-12)
flip_effect = ((scores[best_flip].mean() - scores["as given"].mean())
               / (scores[best].mean() - scores["shuffled poses"].mean() + 1e-12))


def docked(rots, coords):
    """A docked model sits at a local optimum: turning it by 30° about any axis must fit worse. Turns that hardly
    change the projections (about a symmetry axis) can't test this and are skipped."""
    base, p0 = score(rots, coords)
    norm0 = p0.flatten(1).norm(dim=1).numpy() + 1e-12
    worst, informative = -np.inf, 0
    for e in np.eye(3):
        for sign in (1, -1):
            q = Rotation.from_rotvec(np.radians(30.0) * sign * e).as_matrix()
            s, p = score(rots @ q, coords)
            if np.mean(((p - p0).flatten(1).norm(dim=1).numpy() / norm0)[keep]) < 0.1:
                continue
            informative += 1
            d = (s - base)[keep]
            worst = max(worst, d.mean() / (d.std(ddof=1) / np.sqrt(n) + 1e-12) * (-1 if inverted else 1))
    return informative < 2 or worst < -5


if n < 10:
    verdict, choice = "too few", ""
elif not inverted and (scores[best].mean() <= 0 or scores["shuffled poses"].mean()
                       < -5 * scores["shuffled poses"].std(ddof=1) / np.sqrt(n)):
    verdict, choice = "negative", ""  # anti-correlation that the contrast test couldn't pin on the poses
elif not pose_z > 5:
    verdict, choice = "no match", ""
elif pose_frac < 0.3:
    verdict, choice = "weak", ""
elif paired(best_flip, "as given") > 5 and flip_effect > 0.15:
    verdict, choice = "flip", best_flip[-1]
    if paired(best_flip, second_flip) < 5:  # two axes fit about equally: use what we know about the flip
        pair = {best_flip[-1], second_flip[-1]}
        choice = hint if hint in pair else ("z" if "z" in pair else "")
        if not choice:
            verdict = "axis unclear"
elif best == "as given" and paired("as given", best_flip) > 5:
    verdict, choice = "as given", ""
else:
    verdict, choice = "unclear", ""
if verdict == "unclear" and hint and paired(f"flipped along {hint}", "as given") > -5:
    verdict, choice = "flip (map fit)", hint  # the particles can't tell, and don't contradict the map fit
if verdict == "unclear" and paired(best_flip, "as given") > 5:
    verdict = "small flip effect"  # significant, but much smaller than a wrong hand normally makes
if verdict in ("flip", "flip (map fit)", "as given"):
    win = f"flipped along {choice}" if choice else "as given"
    if not docked(*candidates[win]):
        if verdict == "flip (map fit)" and docked(*candidates["as given"]):
            verdict = "map-fit hand not docked"
        else:
            verdict = "not docked"
        choice = ""
auto_choice = choice
if HAND != "auto-detect":
    choice = {"keep the poses as they are": ""}.get(HAND, HAND[-2])
forced = f"flipped along {choice}" if choice else "as given"

rows = ""
for name, s in scores.items():
    z = "" if name == "as given" else f"{paired(name, 'as given'):+.1f}"
    bold = name == forced
    rows += (f"<tr><td>{'<b>' if bold else ''}{name}{'</b>' if bold else ''}</td>"
             f"<td>{s.mean():.4f} ± {s.std(ddof=1) / np.sqrt(max(n, 2)):.4f}</td><td>{z}</td></tr>")
display(HTML(f"<p>Mean correlation of {n:,} particles with the model's projections ({D} px, {apix_d:.2f} Å/px, "
             f"low-passed to {res:.0f} Å{' (limited by the check box)' if res > CHECK_RESOLUTION else ''}"
             + (", sign flipped because the contrast is inverted" if inverted else "") + "):</p>"
             "<table><tr><th>poses</th><th>correlation ± s.e.</th>"
             "<th>paired difference vs as given, in s.e. (auto-detect needs &gt; 5)</th></tr>" + rows + "</table>"))

contradicted = False
if inverted:
    toggle = "Set INVERT_CONTRAST to no" if DATA.get("inverted_contrast") else "Set INVERT_CONTRAST to yes"
    hand_text = {"flip": f"mirrored along {auto_choice}", "flip (map fit)": f"mirrored along {auto_choice}",
                 "as given": "as given"}.get(verdict, "")
    note("The particles have the <b>opposite contrast</b> from what CryoSTAR expects (the poses that fit best "
         f"anti-correlate with the model). {toggle} in 2.1 and run 2.1–2.3 again before training"
         + (f"; the poses look right {hand_text}." if hand_text else ".")
         + ("" if DATA.get("inverted_contrast") else " (cryoDRGN runs that needed --uninvert-data are a typical sign of "
            "this.)") + " The poses are left unchanged for now.", "error")
    choice = ""
elif HAND != "auto-detect" and n >= 10:
    alt = max((k for k in hands if k != forced), key=lambda k: scores[k].mean())
    contradicted = paired(alt, forced) > 5 and pose_z > 5
    if contradicted:
        note(f"HAND forces '{forced}', but the particles fit '{alt}' much better ({paired(alt, forced):+.0f} s.e.). "
             "CryoSTAR's maps would probably be the mirror image of your model; use auto-detect unless you are sure.",
             "warn")

if choice:
    blocks = starfile.read(star_in, always_dict=True)
    key = "particles" if "particles" in blocks else list(blocks)[-1]
    table = blocks[key]
    angles = table[["rlnAngleRot", "rlnAngleTilt", "rlnAnglePsi"]].to_numpy(float)
    R_all = Rotation.from_euler("ZYZ", angles, degrees=True).as_matrix().transpose(0, 2, 1)  # CryoSTAR's matrix
    R_new = MIRROR["z"] @ R_all @ MIRROR[choice]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # gimbal lock: the angles change, the matrix doesn't
        table[["rlnAngleRot", "rlnAngleTilt", "rlnAnglePsi"]] = Rotation.from_matrix(
            R_new.transpose(0, 2, 1)).as_euler("ZYZ", degrees=True)
    tag = hashlib.sha1(str(Path(star_in).resolve()).encode()).hexdigest()[:6]
    out = DATA_ROOT / "hand" / f"{Path(star_in).stem}_{tag}_flipped_{choice}.star"
    out.parent.mkdir(parents=True, exist_ok=True)
    starfile.write(blocks, out, overwrite=True, float_format="%.8f")
    DATA["star"], DATA["hand_flip"] = out, choice
    DATA["pdb"] = pdb_fit if (pdb_fit and choice == hint) else pdb_as_docked
    note(f"Using poses mirrored for a map flipped along {choice} (<code>{out.name}</code>) and the model "
         f"<code>{Path(DATA['pdb']).name}</code>. CryoSTAR's structures and maps will be in the same hand as your model."
         + (" (Chosen from the map fit in 2.2; the particles alone were not decisive.)"
            if verdict == "flip (map fit)" and HAND == "auto-detect" else "")
         + " If you already ran 3.1 or 4.1, run them again.", "ok")
else:
    DATA["star"], DATA["hand_flip"], DATA["pdb"] = star_in, "", pdb_as_docked
    if inverted:
        pass
    elif HAND != "auto-detect":
        if verdict != "as given" and not contradicted:
            note(f"HAND keeps the poses as given. Auto-detect would have said: {verdict}"
                 + (f" (flipped along {auto_choice})" if auto_choice else "") + ".",
                 "info" if verdict in ("unclear", "too few") else "warn")
    elif verdict == "as given":
        note("The poses fit the model as given.", "ok")
    elif verdict == "no match":
        note("The model fits the particles no better with each particle's own pose than with another particle's. "
             "Check that the particle images and the poses are in the same order (for cryoDRGN files: the stack, "
             "pose.pkl/ctf.pkl and IND_PKL), that the model is centred (2.2) and that the pixel size is right. "
             "The poses are kept as given.", "warn")
    elif verdict == "weak":
        note(f"The poses explain only {pose_frac:.0%} of the model's fit to the particles, so the model doesn't match "
             "them well (wrong or very incomplete model, or a pixel-size or centring problem). The poses are kept as "
             "given; check the model before training.", "warn")
    elif verdict == "not docked":
        note(f"The poses fit best {'as given' if best == 'as given' else best}, but turning the model by 30° about "
             "some axis fits the particles about as well or better, so the model doesn't seem to be docked in the "
             "frame of these poses (or belongs to another particle). Check the docking in the consensus map (or its "
             "hand-flipped copy) and the centring (2.2), then run 2.3 again. The poses are kept as given.", "warn")
    elif verdict == "negative":
        note("The model's projections don't correlate positively with the particles for any hand of the poses. The "
             "contrast may be inverted (INVERT_CONTRAST in 2.1) and the model may not be docked in these poses' "
             "frame (2.2). The poses are kept as given.", "warn")
    elif verdict == "axis unclear":
        note(f"Mirroring fits better, but along {best_flip[-1]} and {second_flip[-1]} about equally well. Set HAND to "
             "the axis your map was flipped along. The poses are kept as given for now.", "warn")
    elif verdict == "too few":
        note("Too few particles to decide; set HAND yourself.", "warn")
    elif verdict == "map-fit hand not docked":
        note(f"2.2's map fit suggests that the map was flipped along {hint}, but with the poses mirrored that way, "
             "turning the model by 30° fits the particles about as well, so that hand isn't confirmed (and the "
             "particles alone can't tell the hands apart). Check that 2.2 was given the map these poses produced, "
             "unflipped; set HAND only if you are sure how the map was flipped. The poses are kept as given.", "warn")
    elif verdict == "small flip effect":
        note(f"The poses {best_flip} fit significantly better ({paired(best_flip, 'as given'):+.0f} s.e.), but "
             f"improve the fit only slightly ({flip_effect:.0%} of what the poses explain, less than a wrong hand "
             "normally does), so auto-detect keeps them as given. The hand may show only in a small part of the "
             f"model; if you know the map was flipped along {best_flip[-1]}, set HAND.", "warn")
    else:
        note("Neither hand fits clearly better. Check that the model is centred (2.2); otherwise the particle may "
             "look almost the same in both hands at this resolution: try CHECK_PARTICLES = 3000, or set HAND "
             "yourself. The poses are kept as given.", "warn")
if hint and not inverted and HAND == "auto-detect" and choice != hint and verdict in (
        "as given", "flip", "unclear", "axis unclear", "small flip effect"):
    note(f"2.2's map fit suggested that the map was flipped along {hint}, but the particles don't support that"
         + (f" (they fit the poses {'as given' if not choice else 'flipped along ' + choice} better)" if choice or
            verdict == "as given" else "") + ". Was the map given to 2.2 the hand-flipped copy, or a map from other "
         "particles?", "warn")

DATA["pose_check"] = ("inverted contrast" if inverted
                      else verdict if HAND == "auto-detect" or choice or verdict == "as given"
                      else f"HAND set to keep them; auto-detect: {verdict}")

# A few particles next to the projections with the poses as given and with the chosen (or best) mirror.
show_flip = forced if forced != "as given" else best_flip
idx = np.flatnonzero(keep)[np.linspace(0, n - 1, min(6, n)).astype(int)] if n else []
fig, axes = plt.subplots(3, max(len(idx), 1), figsize=(1.7 * max(len(idx), 1), 5.4), squeeze=False)
for col, i in enumerate(idx):
    for row, (label, img) in enumerate((("particle", images.cpu()), ("as given", projections["as given"]),
                                        (show_flip, projections[show_flip]))):
        axes[row, col].imshow(img[i].numpy() * (-1 if inverted and row == 0 else 1), cmap="gray")
        axes[row, col].set_xticks([]), axes[row, col].set_yticks([])
        if col == 0:
            axes[row, col].set_ylabel(label, fontsize=9)
fig.suptitle(f"Particles (low-passed to {res:.0f} Å{', contrast flipped' if inverted else ''}) "
             "and model projections", fontsize=10)
plt.tight_layout()
plt.show()
del images, projections
if device == "cuda":
    torch.cuda.empty_cache()

## 3 · Stage 1 — train the atom generator

A variational auto-encoder reads every particle image and outputs a small **latent code** *z* (8 numbers by default) plus a **deformed copy of the reference model**, coarse-grained to one bead per residue (Cα for protein, C1′ for nucleic acid). The deformations are kept physically sensible by restraints on chain connectivity, an elastic network and clashes, and are fitted by comparing projections of the beads with the particle images.

Results are saved at every evaluation, in folders named `<epoch>_<step>`; the last one is the final result. The newest one also holds a checkpoint (`ckpt.pt`) from which `CONTINUE_FROM` (3.1) can train the run further.

In [ ]:
#@title 3.1 · Configure stage 1 { display-mode: "form" }
RUN_NAME = "run1"  #@param {type:"string"}
#@markdown A short name for this experiment; outputs go to `work_dirs/atom_colab_<RUN_NAME>…`.
QUICK_TEST = False  #@param {type:"boolean"}
#@markdown Tick for a short run (a few minutes) that checks everything works end to end. The results will be rough.
#@markdown ---
#@markdown **Training length.** `auto` uses the recommended defaults: 24,000 steps for the tutorial, 96,000 for your own data. Pick a value or type your own.
MAX_STEPS = "auto"  #@param ["auto", "12000", "24000", "48000", "96000"] {allow-input: true}
EVALUATE_EVERY = "auto"  #@param ["auto", "3000", "6000", "12000"] {allow-input: true}
#@markdown Each evaluation saves structures, latent codes and a checkpoint.
#@markdown ---
#@markdown **Continue an earlier run** (optional)
CONTINUE_FROM = ""  #@param {type:"string"}
#@markdown To train a stage-1 run further (for example one stopped by a disconnect), give its run folder, one of its `<epoch>_<step>` folders or its `ckpt.pt`. Training starts from its newest checkpoint (stage 1 keeps only that one) in a new run folder, whose step numbers start again at 0. `MAX_STEPS` is then the number of extra steps (`auto`: the steps left of its plan, evaluated as often as before; `LOW_PASS_RESOLUTION = auto` keeps the earlier run's value); `QUICK_TEST` trains 2,000 more steps. `IMAGE_SIZE` and `LATENT_DIM` always come from that run, and 2.1–2.3 must select the data it was trained on. Blank = train from scratch. `USE_EXISTING_RUN` (3.2) takes precedence: it reuses a run without training.
#@markdown ---
#@markdown **Advanced** (the defaults work well)
BATCH_SIZE = 64  #@param {type:"integer"}
IMAGE_SIZE = "auto"  #@param ["auto", "64", "96", "128", "160", "192", "256"] {allow-input: true}
#@markdown Particles are downsampled to this box size (pixels). `auto`: the original size up to 256 px, otherwise 128.
LOW_PASS_RESOLUTION = "auto"  #@param ["auto", "4", "6", "8", "10", "15"] {allow-input: true}
#@markdown Å. Images are low-pass filtered to this resolution before being compared with the bead model (`auto`: 4 Å for the tutorial, 10 Å for your own data).
LATENT_DIM = 8  #@param {type:"integer"}

need(DATA="2.1 · Choose a dataset")
from mmengine import Config

note_pose_status()
base = Config.fromfile(str(STAR_DIR / "atom_configs" / DATA["base"]))
ATOM_CONTINUE = None
if CONTINUE_FROM.strip():
    prev_run, ckpt, prev = continue_point(CONTINUE_FROM, "atom")
    if note_run_inputs(prev_run, "A run can only be trained further on the data it was trained on; to train on the "
                       "data selected now, leave CONTINUE_FROM blank to start a new run."):
        raise RuntimeError(f"{prev_run.name} was trained on different data (see above), so it can't be continued.")
    note_gone_inputs(prev_run)
    if not (ckpt.parent / "z.npy").exists():
        note(f"The newest checkpoint, in <code>{ckpt.parent.name}</code>, was saved when that evaluation started; the "
             "evaluation itself did not finish (there is no z.npy). Training continues from this checkpoint.")
    # Steps trained so far, counting the runs this one continued (colab_continue, written below).
    lineage = prev.get("colab_continue") or {}
    done = int(lineage.get("steps_before", 0)) + int(ckpt.parent.name.split("_")[1])
    planned = int(lineage.get("planned_total", prev.trainer.max_steps))
    ATOM_CONTINUE = dict(from_ckpt=ckpt, from_run=prev_run.resolve(), steps_before=done,
                         warmup_end=int(lineage.get("warmup_end", prev.loss.warmup_step)))
if QUICK_TEST:
    steps, every = 2000, 1000
else:
    if ATOM_CONTINUE:  # MAX_STEPS counts the extra steps; auto = what is left of the earlier run's plan
        steps = planned - done if MAX_STEPS == "auto" else int(MAX_STEPS)
        if steps <= 0:
            raise ValueError(f"{prev_run.name} has already reached the {planned:,} steps it was set up for (counting "
                             "the runs it continues, if any). To train it further, set MAX_STEPS to the number of "
                             "extra steps.")
    else:
        steps = int(base.trainer.max_steps) if MAX_STEPS == "auto" else int(MAX_STEPS)
    if EVALUATE_EVERY != "auto":
        every = int(EVALUATE_EVERY)
    elif MAX_STEPS == "auto":  # the recommended plan, or the rest of the earlier run's plan at its interval
        every = int((prev if ATOM_CONTINUE else base).trainer.val_check_interval)
    else:  # a length of your own, also when continuing
        every = max(steps // 4, 1)
every = max(1, min(every, steps))
if steps % every:
    steps = math.ceil(steps / every) * every
    note(f"Training length rounded up to {steps:,} steps so that the final step is evaluated.")

RUN_TAG = re.sub(r"[^A-Za-z0-9_-]+", "_", RUN_NAME.strip()) or "run"
dataset_attr = dict(dataset_dir=DATA["root"], starfile_path=DATA["star"], ref_pdb_path=DATA["pdb"])
if DATA["apix_override"] or DATA["kind"] == "own":
    dataset_attr["apix"] = DATA["apix"]
data_process = {}
if ATOM_CONTINUE:
    # The checkpoint only fits the network it was saved from, so the settings that shape it come from that run.
    data_process["down_side_shape"] = int(prev.data_process.down_side_shape)
    if IMAGE_SIZE != "auto" and min(int(IMAGE_SIZE), DATA["box"]) != data_process["down_side_shape"]:
        note(f"IMAGE_SIZE: {prev_run.name} was trained on {data_process['down_side_shape']} px images, so that size "
             "is used.")
    if LATENT_DIM != prev.model.model_cfg.latent_dim:
        note(f"LATENT_DIM: {prev_run.name} has {prev.model.model_cfg.latent_dim} latent dimensions, so that is used.")
elif IMAGE_SIZE != "auto":
    data_process["down_side_shape"] = min(int(IMAGE_SIZE), DATA["box"])
if LOW_PASS_RESOLUTION != "auto":
    data_process["low_pass_bandwidth"] = float(LOW_PASS_RESOLUTION)
elif ATOM_CONTINUE and prev.data_process.low_pass_bandwidth != base.data_process.low_pass_bandwidth:
    data_process["low_pass_bandwidth"] = float(prev.data_process.low_pass_bandwidth)
sections = dict(
    exp_name="",
    dataset_attr=dataset_attr,
    data_loader=dict(train_batch_per_gpu=BATCH_SIZE, workers_per_gpu=min(4, os.cpu_count())),
    trainer=dict(max_steps=steps, val_check_interval=every, devices=1),
)
if data_process:
    sections["data_process"] = data_process
if ATOM_CONTINUE:
    for key in ("model", "gmm"):
        if config_changes(prev[key], base[key]):
            sections[key] = config_changes(prev[key], base[key])
    # Load the checkpoint and skip CryoSTAR's initial fit to the reference model (do_ref_init), which would undo it.
    # The KL weight rises linearly from 0 over loss.warmup_step steps; ending the rise at the same total step as the
    # earlier run keeps it at or below that run's schedule (1 = full weight at once, when the rise was over).
    sections.update(do_ref_init=False, extra_input_data_attr=dict(ckpt_path=ckpt),
                    loss=dict(warmup_step=max(1, ATOM_CONTINUE["warmup_end"] - done)))
    ATOM_CONTINUE["planned_total"] = done + steps
    sections["colab_continue"] = ATOM_CONTINUE  # read back when this run is continued in turn
elif LATENT_DIM != base.model.model_cfg.latent_dim:
    sections["model"] = dict(model_cfg=dict(latent_dim=LATENT_DIM))
ATOM_STEPS, ATOM_INPUTS = steps, data_inputs("star", "pdb", "root", "apix")
ATOM_CONFIG = write_config(STAR_DIR / "atom_configs" / f"colab_{RUN_TAG}.py", DATA["base"], **sections)

print(f"Config written to {ATOM_CONFIG} (it inherits atom_configs/{DATA['base']}):\n")
print(ATOM_CONFIG.read_text())
if ATOM_CONTINUE:
    own = int(ckpt.parent.name.split("_")[1])  # the steps of prev_run itself
    after = f"{done:,} steps" + (f" in total ({own:,} of them in {prev_run.name})" if own != done else "")
    note(f"Stage 1 continues from <code>{ckpt}</code> after {after} and will train <b>{steps:,} more</b> "
         f"({done + steps:,} in total) with batch size {BATCH_SIZE}, evaluating every {every:,} steps. Its new run "
         "folder counts steps from 0.")
else:
    note(f"Stage 1 will train for <b>{steps:,} steps</b> with batch size {BATCH_SIZE}, "
         f"evaluating every {every:,} steps.")

In [ ]:
#@title 3.2 · Train stage 1 { display-mode: "form" }
#@markdown A progress bar with the estimated time left appears below, followed by the training log. You can stop early with ■; results from evaluations that already finished stay usable.
#@markdown
#@markdown To reuse a finished run instead of training (for example after a disconnect), give its folder. This takes precedence over `CONTINUE_FROM` in 3.1, which trains a run further:
USE_EXISTING_RUN = ""  #@param {type:"string"}

need(ATOM_CONFIG="3.1 · Configure stage 1")
if USE_EXISTING_RUN:
    if globals().get("ATOM_CONTINUE"):
        note("USE_EXISTING_RUN is set, so nothing is trained and CONTINUE_FROM (3.1) is not used.")
    run = run_folder(USE_EXISTING_RUN)
    note_run_inputs(run, "3.3 and stage 2 use the current selection, so they would not match this run.")
else:
    if not HAS_GPU:
        raise RuntimeError("No GPU: choose Runtime ▸ Change runtime type ▸ T4 GPU, then run the notebook again.")
    if DATA.get("pose_check") == "inverted contrast":
        raise RuntimeError("2.3 found that the particles' contrast is inverted. Change INVERT_CONTRAST in 2.1 and "
                           "run 2.1–2.3 again first.")
    if ATOM_INPUTS != data_inputs("star", "pdb", "root", "apix"):
        raise RuntimeError("The dataset, poses or model changed after 3.1 was run (2.1, 2.2 and 2.3 can change "
                           "them). If you re-ran 2.1 or 2.2, run 2.3 again; then run 3.1 and this cell.")
    run = run_stage("Stage 1 · atom generator", "train_atom.py", ATOM_CONFIG, max_steps=ATOM_STEPS)
    if run is None:
        raise RuntimeError("Training was stopped before CryoSTAR created its run folder, so there are no new results.")

# ATOM_RUN and ATOM_RESULT change together, only once the run has results (3.3 onward use both).
finished = result_dirs(run, "z.npy")
if not finished:
    raise RuntimeError(f"No evaluation has finished in {run} yet, so there are no results to use.")
ATOM_RUN, ATOM_RESULT = run, finished[-1]
note(f"Stage 1 results: <code>{ATOM_RESULT.resolve()}</code>", "ok")

In [ ]:
#@title 3.3 · Training curves and image check { display-mode: "form" }
#@markdown Top: losses during training. Below: particle images next to projections of the predicted bead models for the same particles. The projections are low-pass filtered and include the CTF, so they look smoother and show ripples, but should match each particle's overall shape and orientation. Blue marks the area outside the mask used by the loss.
need(ATOM_RESULT="3.2 · Train stage 1")
import matplotlib.pyplot as plt

note_pose_status()
note_run_inputs(ATOM_RUN, "The status above describes the current selection, not this run.")
lineage = (run_config(ATOM_RUN) or {}).get("colab_continue")
if lineage:
    before, own = int(lineage["steps_before"]), int(Path(str(lineage["from_ckpt"])).parent.name.split("_")[1])
    note(f"This run continues <code>{Path(lineage['from_run']).name}</code> after {before:,} steps"
         + (f" in total ({own:,} of them in that run)" if own != before else "")
         + "; the steps below count from the start of this continuation.")
NAMES = {"loss": "Total loss", "cryoem(gmm)": "Image term (−correlation)", "con": "Chain connectivity",
         "dist": "Elastic network", "clash": "Clashes", "kld": "KL divergence (weighted)"}
metrics = {k: v for k, v in read_metrics(ATOM_RUN).items() if k in NAMES and any(v[1])}
if metrics:
    fig, axes = plt.subplots(1, len(metrics), figsize=(3.2 * len(metrics), 2.6), squeeze=False)
    for ax, (key, (x, y)) in zip(axes[0], metrics.items()):
        ax.plot(x, y, color=SERIES_BLUE, linewidth=1.5)
        ax.set_title(NAMES[key], fontsize=10)
        ax.set_xlabel("step", fontsize=9)
        style_axes(ax)
    plt.tight_layout()
    plt.show()
else:
    print("No loss values found in the log yet.")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, name, title in zip(axes, ("input_image.png", "pred_gmm_image.png"),
                           ("Particle images", "Projections of the predicted bead models")):
    ax.imshow(plt.imread(ATOM_RESULT / name))
    ax.set_title(title, fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
#@title 3.4 · Latent space { display-mode: "form" }
#@markdown Every particle has a latent code *z*. Here they are projected onto their first principal components (PCs); clusters or elongated clouds indicate distinct states or continuous motions. The structures in 3.5 and the maps in section 4 are sampled along PC1–PC3.
RUN_UMAP = False  #@param {type:"boolean"}
#@markdown Also show a UMAP embedding (takes a minute or two for large datasets).

need(ATOM_RESULT="3.2 · Train stage 1")
import numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA

z = np.load(ATOM_RESULT / "z.npy")
pca = PCA(min(z.shape[1], 3)).fit(z)
pc = pca.transform(z)
panels = [(pc[:, 0], pc[:, 1], "PC1", "PC2")]
if pc.shape[1] > 2:
    panels.append((pc[:, 0], pc[:, 2], "PC1", "PC3"))
if RUN_UMAP:
    from cryostar.utils.latent_space_utils import run_umap
    emb, _ = run_umap(z)
    panels.append((emb[:, 0], emb[:, 1], "UMAP 1", "UMAP 2"))

fig, axes = plt.subplots(1, len(panels), figsize=(4.4 * len(panels), 4), squeeze=False)
for ax, (x, y, xl, yl) in zip(axes[0], panels):
    hb = ax.hexbin(x, y, gridsize=60, cmap=blue_cmap(), mincnt=1, bins="log")
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    style_axes(ax)
    ax.grid(False)
fig.colorbar(hb, ax=axes[0].tolist(), label="particles per bin (log scale)", shrink=0.85)
plt.show()
ratios = ", ".join(f"PC{i + 1} {r:.0%}" for i, r in enumerate(pca.explained_variance_ratio_))
print(f"{len(z):,} particles, latent dimension {z.shape[1]}. Variance explained: {ratios}.")

In [ ]:
#@title 3.5 · Structures along the principal components { display-mode: "form" }
#@markdown Ten bead models sampled from the 5th to the 95th percentile along a PC, played as a movie. Drag to rotate, scroll to zoom.
TRAJECTORY = "PC1"  #@param ["PC1", "PC2", "PC3", "k-means cluster centres"]
STYLE = "cartoon"  #@param ["cartoon", "spheres"]
SHOW_EXTREMES = True  #@param {type:"boolean"}
#@markdown Also show the first (blue) and last (orange) structure superimposed, to see the range of motion.

need(ATOM_RESULT="3.2 · Train stage 1")
import py3Dmol

fname = {"PC1": "pca-1.pdb", "PC2": "pca-2.pdb", "PC3": "pca-3.pdb",
         "k-means cluster centres": "pred.pdb"}[TRAJECTORY]
pdb_text = (ATOM_RESULT / fname).read_text()
frames = split_models(pdb_text)
look = (lambda color: {"cartoon": {"color": color, "style": "trace", "thickness": 0.8}}) if STYLE == "cartoon" \
    else (lambda color: {"sphere": {"color": color, "radius": 1.6}})

view = py3Dmol.view(width=720, height=460)
view.addModelsAsFrames(pdb_text, "pdb")
view.setStyle(look("spectrum"))
view.animate({"loop": "backAndForth", "interval": 250})
view.zoomTo()
print(f"{TRAJECTORY}: {len(frames)} structures from {ATOM_RESULT / fname}")
view.show()

if SHOW_EXTREMES and len(frames) > 1:
    view = py3Dmol.view(width=720, height=460)
    view.addModel(frames[0], "pdb")
    view.setStyle({"model": 0}, look(SERIES_BLUE))
    view.addModel(frames[-1], "pdb")
    view.setStyle({"model": 1}, look(SERIES_ORANGE))
    view.zoomTo()
    view.show()

## 4 · Stage 2 — train the density generator

Stage 2 keeps the latent code *z* that stage 1 assigned to each particle and trains a neural density model that turns any *z* into a 3D map. At the end it writes maps sampled along PC1–PC3 (`vol_pca_<pc>_<frame>.mrc`) and at k-means centres (`vol_kmeans_<i>.mrc`).

In [ ]:
#@title 4.1 · Configure stage 2 { display-mode: "form" }
QUICK_TEST = False  #@param {type:"boolean"}
#@markdown Tick for a short run (a few minutes) that checks everything works end to end. The maps will be rough.
#@markdown ---
MAX_EPOCHS = "auto"  #@param ["auto", "5", "10", "20"] {allow-input: true}
#@markdown Passes over all particles. `auto`: 5 for the tutorial, 20 for your own data.
#@markdown ---
#@markdown **Continue an earlier run** (optional)
CONTINUE_FROM = ""  #@param {type:"string"}
#@markdown To train a stage-2 run further, give its run folder (for its newest checkpoint), one of its `<epoch>_<step>` folders or one of its `ckpt.pt` files. Training continues in a new run folder, whose epochs and steps start again at 0; `MAX_EPOCHS` is then the number of extra epochs (`auto`: the epochs left of the earlier run's plan; `QUICK_TEST`: one short epoch). `IMAGE_SIZE` and `DECODER_WIDTH` come from that run. It needs the same particles and latent codes as that run (select them with `USE_EXISTING_RUN` in 3.2 or with `Z_FILE`), so once stage 1 has been trained further (which gives new codes), train stage 2 from scratch. Blank = train from scratch. `USE_EXISTING_RUN` (4.2) takes precedence.
#@markdown ---
#@markdown **Advanced** (the defaults work well)
BATCH_SIZE = 16  #@param {type:"integer"}
IMAGE_SIZE = "auto"  #@param ["auto", "64", "96", "128", "160", "192", "256"] {allow-input: true}
#@markdown Box size (pixels) of the output maps. `auto`: 128 px, or the original size if smaller. Larger boxes need much more time and GPU memory.
DECODER_WIDTH = "1024"  #@param ["256", "512", "1024"]
#@markdown Width of the density network. Smaller is faster but less detailed.
Z_FILE = ""  #@param {type:"string"}
#@markdown Latent codes to use. Blank = `z.npy` from stage 1 (3.2).

need(DATA="2.1 · Choose a dataset")
import numpy as np
from mmengine import Config

note_pose_status()
if Z_FILE:
    z_path = Path(Z_FILE.strip()).expanduser().resolve()
    z_run = z_path.parent.parent  # <run>/<epoch>_<step>/z.npy
else:
    need(ATOM_RESULT="3.2 · Train stage 1")
    z_path, z_run = ATOM_RESULT / "z.npy", ATOM_RUN
if note_run_inputs(z_run, "Stage 2 must use the same particles, poses and pixel size as stage 1.", ignore=("model",)):
    raise RuntimeError("The latent codes come from a stage-1 run trained on different data (see above).")
z = np.load(z_path, mmap_mode="r")
if len(z) != DATA["n"]:
    raise ValueError(f"{z_path} has codes for {len(z):,} particles but the dataset has {DATA['n']:,}.")

base = Config.fromfile(str(STAR_DIR / "density_configs" / DATA["base"]))
DENSITY_CONTINUE = None
if CONTINUE_FROM.strip():
    prev_run, ckpt, prev = continue_point(CONTINUE_FROM, "density")
    if note_run_inputs(prev_run, "A run can only be trained further on the data it was trained on; to train on the "
                       "data selected now, leave CONTINUE_FROM blank to start a new run.", ignore=("model",)):
        raise RuntimeError(f"{prev_run.name} was trained on different data (see above), so it can't be continued.")
    note_gone_inputs(prev_run)
    # It must also get the codes it was trained on. Each of its evaluations saved a copy of them as z.npy.
    trained_z = Path(str(prev.extra_input_data_attr.given_z))
    copies = result_dirs(prev_run, "z.npy")
    if not (copies or trained_z.is_file()):
        note(f"The latent codes {prev_run.name} was trained on (<code>{trained_z}</code>) are no longer here, so they "
             f"can't be compared with <code>{z_path}</code>. Continue only with the same codes.", "warn")
    elif not same_codes(copies[-1] / "z.npy" if copies else trained_z, z_path):
        z_lineage = (run_config(z_run) or {}).get("colab_continue")
        raise RuntimeError(
            f"{prev_run.name} was trained on other latent codes ({trained_z}) than the ones selected now ({z_path})"
            + (f"; {z_run.name} trained stage 1 further from {Path(z_lineage['from_run']).name}, which changes every "
               "code" if z_lineage else "")
            + ". Stage 2 can only continue with the same codes: leave CONTINUE_FROM blank to train stage 2 from "
            "scratch on the codes selected now" + (f", or set Z_FILE to {trained_z}." if trained_z.is_file() else "."))
    # Epochs trained so far, counting the runs this one continued (colab_continue, written below). Results are saved
    # at the end of an epoch, in folders named after its (0-based) number.
    lineage = prev.get("colab_continue") or {}
    done = int(lineage.get("epochs_before", 0)) + int(ckpt.parent.name.split("_")[0]) + 1
    planned = int(lineage.get("planned_total", prev.trainer.max_epochs))
    DENSITY_CONTINUE = dict(from_ckpt=ckpt, from_run=prev_run.resolve(), epochs_before=done)
if DENSITY_CONTINUE and not QUICK_TEST and MAX_EPOCHS == "auto":  # what is left of the earlier run's plan
    epochs = planned - done
    if epochs <= 0:
        raise ValueError(f"{prev_run.name} has already reached the {planned} epochs it was set up for (counting the "
                         "runs it continues, if any). To train it further, set MAX_EPOCHS to the number of extra "
                         "epochs.")
else:
    epochs = 1 if QUICK_TEST else (int(base.trainer.max_epochs) if MAX_EPOCHS == "auto" else int(MAX_EPOCHS))
plan = prev if DENSITY_CONTINUE and MAX_EPOCHS == "auto" else base  # the rest of the earlier plan, or a new length
every = min(int(plan.trainer.check_val_every_n_epoch or epochs), epochs)
if epochs % every:
    epochs = math.ceil(epochs / every) * every
    note(f"Training length rounded up to {epochs} epochs so that the final epoch is evaluated.")
network = dict(hidden=int(DECODER_WIDTH), z_dim=int(z.shape[1]))
if DENSITY_CONTINUE:
    # The checkpoint only fits the network it was saved from, so the settings that shape it come from that run.
    size = int(prev.data_process.down_side_shape)
    if IMAGE_SIZE != "auto" and min(int(IMAGE_SIZE), DATA["box"]) != size:
        note(f"IMAGE_SIZE: {prev_run.name} makes {size}×{size} px maps, so that size is used.")
    network = dict(hidden=int(prev.model.hidden), z_dim=int(prev.model.z_dim))
    network.update((k, v) for k, v in config_changes(prev.model, base.model).items() if k not in network)
    if int(DECODER_WIDTH) != network["hidden"]:
        note(f"DECODER_WIDTH: {prev_run.name} has a decoder width of {network['hidden']}, so that is used.")
elif IMAGE_SIZE == "auto":
    size = min(base.data_process.down_side_shape or 128, DATA["box"])
else:
    size = min(int(IMAGE_SIZE), DATA["box"])

dataset_attr = dict(dataset_dir=DATA["root"], starfile_path=DATA["star"])
if DATA["apix_override"] or DATA["kind"] == "own":
    dataset_attr["apix"] = DATA["apix"]
trainer = dict(max_epochs=epochs, check_val_every_n_epoch=every, devices=1)
batches = math.ceil(DATA["n"] / BATCH_SIZE)
if QUICK_TEST:
    trainer["limit_train_batches"] = batches = min(500, batches)

RUN_TAG = globals().get("RUN_TAG", "run")
extra_inputs, more = dict(given_z=z_path), {}
if DENSITY_CONTINUE:
    extra_inputs["ckpt_path"] = ckpt
    DENSITY_CONTINUE["planned_total"] = done + epochs
    more["colab_continue"] = DENSITY_CONTINUE  # read back when this run is continued in turn
DENSITY_EPOCHS, DENSITY_INPUTS = epochs, data_inputs("star", "root", "apix")
DENSITY_CONFIG = write_config(
    STAR_DIR / "density_configs" / f"colab_{RUN_TAG}.py", DATA["base"],
    exp_name="",
    dataset_attr=dataset_attr,
    extra_input_data_attr=extra_inputs,
    data_process=dict(down_side_shape=int(size)),
    data_loader=dict(train_batch_per_gpu=BATCH_SIZE, workers_per_gpu=min(4, os.cpu_count())),
    model=network,
    trainer=trainer,
    **more,
)
print(f"Config written to {DENSITY_CONFIG} (it inherits density_configs/{DATA['base']}):\n")
print(DENSITY_CONFIG.read_text())
if DENSITY_CONTINUE:
    own = int(ckpt.parent.name.split("_")[0]) + 1  # the epochs of prev_run itself
    after = f"{done} epoch(s)" + (f" in total ({own} of them in {prev_run.name})" if own != done else "")
    note(f"Stage 2 continues from <code>{ckpt}</code> after {after} and will train <b>{epochs} more</b> "
         f"({done + epochs} in total; about {batches * epochs:,} steps) on {size}×{size} px images, evaluating every "
         f"{every} epoch(s). Its new run folder counts epochs and steps from 0.")
else:
    note(f"Stage 2 will train for <b>{epochs} epoch(s)</b> (about {batches * epochs:,} steps) "
         f"on {size}×{size} px images, evaluating every {every} epoch(s).")

In [ ]:
#@title 4.2 · Train stage 2 { display-mode: "form" }
#@markdown To reuse a finished run instead of training, give its folder. This takes precedence over `CONTINUE_FROM` in 4.1:
USE_EXISTING_RUN = ""  #@param {type:"string"}

need(DENSITY_CONFIG="4.1 · Configure stage 2")
if USE_EXISTING_RUN:
    if globals().get("DENSITY_CONTINUE"):
        note("USE_EXISTING_RUN is set, so nothing is trained and CONTINUE_FROM (4.1) is not used.")
    run = run_folder(USE_EXISTING_RUN)
    note_run_inputs(run, "The maps in 4.3 and 4.4 come from that data, not the current selection.",
                    ignore=("model",))
else:
    if not HAS_GPU:
        raise RuntimeError("No GPU: choose Runtime ▸ Change runtime type ▸ T4 GPU, then run the notebook again.")
    if DATA.get("pose_check") == "inverted contrast":
        raise RuntimeError("2.3 found that the particles' contrast is inverted. Change INVERT_CONTRAST in 2.1 and "
                           "run 2.1–2.3 again first.")
    if DENSITY_INPUTS != data_inputs("star", "root", "apix"):
        raise RuntimeError("The dataset or poses changed after 4.1 was run (2.1, 2.2 and 2.3 can change them). If "
                           "you re-ran 2.1 or 2.2, run 2.3 again; then run 4.1 and this cell. 4.1 says if stage 1 "
                           "was trained on different data.")
    run = run_stage("Stage 2 · density generator", "train_density.py", DENSITY_CONFIG,
                    max_epochs=DENSITY_EPOCHS)
    if run is None:
        raise RuntimeError("Training was stopped before CryoSTAR created its run folder, so there are no new results.")

# DENSITY_RUN and DENSITY_RESULT change together, only once the run has maps (4.3 onward use both).
finished = result_dirs(run, "vol_pca_1_000.mrc")
if not finished:
    raise RuntimeError(f"No evaluation has finished in {run} yet, so there are no maps to show.")
DENSITY_RUN, DENSITY_RESULT = run, finished[-1]
note(f"Stage 2 results: <code>{DENSITY_RESULT.resolve()}</code>", "ok")

In [ ]:
#@title 4.3 · Maps along a principal component { display-mode: "form" }
#@markdown Projections of the ten maps sampled along one PC (frame 0 → 9), viewed down three axes. Compare columns to see what moves.
PC = "PC1"  #@param ["PC1", "PC2", "PC3"]

need(DENSITY_RESULT="4.2 · Train stage 2")
import numpy as np, mrcfile, matplotlib.pyplot as plt

k = PC[-1]
paths = sorted(DENSITY_RESULT.glob(f"vol_pca_{k}_*.mrc"))
if not paths:
    raise FileNotFoundError(f"No maps for {PC} in {DENSITY_RESULT}.")
vols = [mrcfile.read(p) for p in paths]
fig, axes = plt.subplots(3, len(vols), figsize=(1.45 * len(vols), 4.6))
for row, axis in enumerate((0, 1, 2)):
    projs = [v.sum(axis) for v in vols]
    lo, hi = np.percentile(np.stack(projs), [1, 99.8])
    for col, proj in enumerate(projs):
        ax = axes[row, col]
        ax.imshow(proj, cmap="gray", vmin=lo, vmax=hi)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(str(col), fontsize=9)
    axes[row, 0].set_ylabel(("down z", "down y", "down x")[row], fontsize=9)
fig.suptitle(f"Density maps along {PC} (frame number on top)", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
#@title 4.4 · 3D movie of the maps { display-mode: "form" }
#@markdown An isosurface of each map along the chosen PC. Press ▶ or drag the slider to step through the frames; drag the view to rotate.
PC = "PC1"  #@param ["PC1", "PC2", "PC3"]
SURFACE_LEVEL = 1.0  #@param {type:"slider", min:0.3, max:3.0, step:0.1}
#@markdown Threshold, as a multiple of the volume expected from the reference model's mass. Lower it to see weaker density.

need(DENSITY_RESULT="4.2 · Train stage 2")
import numpy as np, mrcfile, plotly.graph_objects as go
from skimage.measure import marching_cubes

paths = sorted(DENSITY_RESULT.glob(f"vol_pca_{PC[-1]}_*.mrc"))
with mrcfile.open(paths[0]) as m:
    vs = float(m.voxel_size.x)
vols = [mrcfile.read(p).astype(np.float32) for p in paths]
bin_by = 2 if vols[0].shape[0] > 100 else 1  # keep the meshes light enough for the browser
if bin_by > 1:
    n = vols[0].shape[0] // bin_by * bin_by
    vols = [v[:n, :n, :n].reshape(n // 2, 2, n // 2, 2, n // 2, 2).mean((1, 3, 5)) for v in vols]
    vs *= bin_by

# Threshold so the surface encloses SURFACE_LEVEL × the expected molecular volume
# (~110 Da per residue, ~330 Da per nucleotide, 1.21 Å^3 per Da).
expected = (DATA["n_aa"] * 110 + DATA["n_nt"] * 330) * 1.21 / vs ** 3
frac = float(np.clip(expected * SURFACE_LEVEL / vols[0].size, 1e-4, 0.5))
level = float(np.quantile(vols[len(vols) // 2], 1 - frac))


def mesh(v):
    if not v.min() < level < v.max():
        return go.Mesh3d(x=[], y=[], z=[])
    verts, faces, _, _ = marching_cubes(v, level)
    xyz = verts[:, ::-1] * vs  # (z, y, x) voxels -> (x, y, z) Å
    return go.Mesh3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                     color=SERIES_BLUE, opacity=1.0, flatshading=False,
                     lighting=dict(ambient=0.45, diffuse=0.8, specular=0.2), hoverinfo="skip")


meshes = [mesh(v) for v in vols]
extent = vols[0].shape[0] * vs
axis = dict(range=[0, extent], visible=False)
steps = [dict(method="animate", label=str(i),
              args=[[str(i)], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
         for i in range(len(meshes))]
fig = go.Figure(
    data=[meshes[0]],
    frames=[go.Frame(data=[m], name=str(i)) for i, m in enumerate(meshes)],
    layout=go.Layout(
        title=f"Density maps along {PC}", height=560, margin=dict(l=0, r=0, t=40, b=0),
        scene=dict(xaxis=axis, yaxis=axis, zaxis=axis, aspectmode="cube"),
        updatemenus=[dict(type="buttons", x=0.02, y=0.02, xanchor="left", yanchor="bottom",
                          buttons=[dict(label="▶ Play", method="animate",
                                        args=[None, dict(frame=dict(duration=350, redraw=True),
                                                         fromcurrent=True, mode="immediate")])])],
        sliders=[dict(active=0, steps=steps, x=0.15, len=0.8, y=0.02,
                      currentvalue=dict(prefix="frame "))],
    ),
)
fig.show()

## 5 · Save the results

In [ ]:
#@title 5.1 · Download the results { display-mode: "form" }
#@markdown `final results only` packs the last evaluation of each stage plus its config and logs; `complete run folders` packs everything, including intermediate evaluations and, for a run trained further with `CONTINUE_FROM`, the earlier run(s) it continues (much larger).
WHAT = "final results only"  #@param ["final results only", "complete run folders"]

assert "need" in globals(), "Please run cell 1.1 first."
files_to_pack, gone = collect_results(WHAT)
if not files_to_pack:
    raise RuntimeError("Nothing to save yet: train stage 1 (3.2) first.")
if gone:
    note(f"Not packed under inputs/ because they no longer exist here: {', '.join(gone)}.", "warn")

zip_path = CONTENT / f"cryostar_{globals().get('RUN_TAG', 'run')}_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f, arcname in files_to_pack:
        zf.write(f, arcname)
print(f"Packed {zip_path.name}: {zip_path.stat().st_size / 1e6:.0f} MB")
if PERSIST_DIR is not None:
    note(f"Your runs are also saved in Google Drive under <code>{PERSIST_DIR / 'work_dirs'}</code>.", "ok")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print(f"Saved to {zip_path}")

In [ ]:
#@title 5.2 · (optional) Save the results to Google Drive { display-mode: "form" }
#@markdown Copies the same files as 5.1 (each stage's final results with its config and logs, and under `inputs/` the model and `.star` file stage 1 was trained with) to a folder in your Google Drive, with no download. The folder is named after `RUN_NAME` (3.1); running the cell again for the same runs copies only what changed, and other runs saved under the same name go to a numbered folder (`…_results_2`). With 1.2 on, your runs are already in Drive: this makes a second, tidy copy, which also counts against your Drive storage.
DRIVE_RESULTS_FOLDER = "CryoSTAR/results"  #@param {type:"string"}
#@markdown A folder in My Drive, created if needed.
WHAT = "final results only"  #@param ["final results only", "complete run folders"]
WAIT_FOR_UPLOAD = False  #@param {type:"boolean"}
#@markdown Tick this before you close Colab: the cell then waits until Google Drive has uploaded everything, and disconnects Drive (run 1.2 again to keep working with Drive).

assert "need" in globals(), "Please run cell 1.1 first."
parts = [p for p in DRIVE_RESULTS_FOLDER.strip().split("/") if p not in ("", ".")]
if DRIVE_RESULTS_FOLDER.strip().startswith("/") and parts[:3] != ["content", "drive", "MyDrive"]:
    raise ValueError(f"DRIVE_RESULTS_FOLDER must be a folder in My Drive, such as CryoSTAR/results, not {DRIVE_RESULTS_FOLDER}.")
for prefix in (["content", "drive", "MyDrive"], ["drive", "MyDrive"], ["MyDrive"], ["My Drive"]):
    if parts[:len(prefix)] == prefix:
        parts = parts[len(prefix):]
        break
if ".." in parts or parts[:1] == ["~"]:
    raise ValueError(f"DRIVE_RESULTS_FOLDER must be a folder in My Drive, such as CryoSTAR/results, not {DRIVE_RESULTS_FOLDER}.")
if not IN_COLAB:
    note("Saving to Google Drive only works inside Colab; use 5.1 to pack the results instead.", "warn")
else:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    files_to_copy, gone = collect_results(WHAT)
    if not files_to_copy:
        raise RuntimeError("Nothing to save yet: train stage 1 (3.2) first.")
    base = Path("/content/drive/MyDrive").joinpath(*parts) / f"cryostar_{globals().get('RUN_TAG', 'run')}_results"
    tops, dest, n = {rel.parts[0] for _, rel in files_to_copy}, base, 1
    while dest.exists() and not {p.name for p in dest.iterdir()} <= tops:  # holds other runs saved under this name
        n += 1
        dest = base.with_name(f"{base.name}_{n}")
    if PERSIST_DIR is not None:
        note(f"Your runs are already in Google Drive under <code>{PERSIST_DIR / 'work_dirs'}</code>; "
             "this cell makes a second copy of the selected files.")
    from tqdm.auto import tqdm
    total = sum(f.stat().st_size for f, _ in files_to_copy)
    copied = unchanged = 0
    with tqdm(total=total, unit="B", unit_scale=True) as bar:
        for f, rel in files_to_copy:
            target, size = dest / rel, f.stat().st_size
            if target.exists() and (target.resolve() == f.resolve() or (
                    target.stat().st_size == size and abs(target.stat().st_mtime - f.stat().st_mtime) <= 1)):
                unchanged += 1  # copied by an earlier run of this cell and not changed since
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(f, target)
                try:
                    shutil.copystat(f, target)
                except OSError:
                    pass
                copied += 1
            bar.update(size)
    if gone:
        note(f"Not copied under inputs/ because they no longer exist here: {', '.join(gone)}.", "warn")
    if dest != base:
        note(f"<code>{base}</code> holds other runs saved under the name {base.name}, so these went to a new folder.")
    size_text = f"{total / 1e9:,.1f} GB" if total > 1e8 else f"{total / 1e6:,.0f} MB"
    note(f"Copied to <code>{dest}</code> ({len(files_to_copy)} files, {size_text}"
         + (f"; {unchanged} were already there unchanged" if unchanged else "") + ").", "ok")
    if WAIT_FOR_UPLOAD:
        note("Waiting for Google Drive to finish uploading…")
        drive.flush_and_unmount()
        note("Everything is in Google Drive, and Drive is now disconnected from this runtime"
             + (": run 1.2 again before using the other cells." if PERSIST_DIR is not None else "."), "ok")
    else:
        note("Google Drive uploads the files in the background. Before you close Colab or delete the runtime, tick "
             "<code>WAIT_FOR_UPLOAD</code> and run this cell again: it copies nothing new and waits until the upload "
             "has finished.")

### Looking at the results in ChimeraX

Inside the `.zip` (5.1) or the Drive folder (5.2), `atom_…/<epoch>_<step>/` holds the stage-1 results, `density_…/<epoch>_<step>/` the maps, and `inputs/` the model and `.star` file used for training.

| File | What it is |
|---|---|
| `pca-1.pdb`, `pca-2.pdb`, `pca-3.pdb` | 10 bead models each, along PC1–PC3 of the latent space |
| `pred.pdb` | bead models at the k-means cluster centres |
| `z.npy` | the latent code of every particle (particles × latent dimension), in `.star` order |
| `vol_pca_<pc>_<frame>.mrc` | density maps along each PC; `z_pca_<pc>.txt` holds the matching codes |
| `vol_kmeans_<i>.mrc` | density maps at k-means cluster centres |
| `ckpt.pt` | model weights |

In [ChimeraX](https://www.cgl.ucsf.edu/chimerax/):

* bead models as a movie: `open pca-1.pdb coordsets true`, then `coordset slider #1`
* maps as a movie: `open vol_pca_1_*.mrc vseries true`, then `vseries slider #2`

Everything is in CryoSTAR's frame, with the origin at the centre of the box, so the beads, the maps and the model in `inputs/` overlay directly. To bring in your consensus map (origin at its corner), give it the same frame with `volume #3 originIndex 64,64,64` (half its box size in pixels). If 2.3 mirrored the poses, the results are in your model's hand, so compare them with the hand-flipped map you docked into; a map flipped by reversing its array (ChimeraX `volume flip`, cryoDRGN `flip_hand`) sits about one voxel off along the flip axis, which is why `inputs/` may hold a `_mirrorfit_` copy of the model.

## Troubleshooting

* **`CUDA out of memory`** — lower `BATCH_SIZE` or `IMAGE_SIZE` in 3.1 / 4.1 and run the training cell again.
* **The runtime disconnected during training.** Turn on Google Drive (1.2) before training. After reconnecting, run 1.1 → 1.3, then 2.1, 2.2 and 2.3 with the same settings as before (so that the model and any mirrored poses are the same again). The stage-1 run folder is `/content/drive/MyDrive/<DRIVE_FOLDER>/work_dirs/atom_colab_<RUN_NAME>_<n>`. To use the results it saved, run 3.1 and give that folder (or one of its result folders) in `USE_EXISTING_RUN` (3.2); stage 2 then starts from its `z.npy`. To finish the training instead, give the folder in `CONTINUE_FROM` (3.1) with `MAX_STEPS` = `auto`, and run 3.1 and 3.2: training continues from the run's newest checkpoint, in a new run folder. Either way, 3.1 (when continuing) or 3.2 (when reusing) says if that run was trained with a different model, poses, pixel size or contrast; 3.1 also says when it can't tell. To continue a stage-2 run (`…/work_dirs/density_colab_<RUN_NAME>_<n>`), first select the latent codes it was trained on: run 3.1 and give its stage-1 run in `USE_EXISTING_RUN` (3.2), or set `Z_FILE` (4.1) to that `z.npy`. Then give the stage-2 run folder in `CONTINUE_FROM` (4.1) and run 4.1 and 4.2.
* **`WARNING: Particle image … invalid! Setting to zeros.`** — the image paths in the `.star` file don't resolve. Set `PARTICLES_ROOT` in 2.1 to the folder they are relative to (for cryoDRGN files, check the stack chosen in 2.1).
* **The bead models look distorted or the projections in 3.3 don't match the particles.** Check that the model is docked in the consensus map and centred (2.2), and that the pixel size is right.
* **cryoDRGN inputs.** The stack's rows must be in the same order as the `.star`/`.cs` the pkls were parsed from (`cryodrgn downsample` keeps that order). With cryoDRGN < 4.3, `parse_pose_csparc` needed `-D`, which must be the box of the particles in the `.cs` file (not a downsampled size); in every version, poses from a heterogeneous refinement need `--hetrefine`. If cryoSPARC aligned the particles at a different pixel size than the stored images (`alignments3D/psize_A` ≠ `blob/psize_A`, e.g. particles downsampled in cryoSPARC before v3.0), parse_pose_csparc mis-scales the shifts; 2.1 checks this when the `.cs` file is in `CRYODRGN_FOLDER`. Poses refined by cryoDRGN itself (`--do-pose-sgd`) work, with shifts about 1/D too small.
* **Wrong hand.** If the consensus map (or cryoDRGN's volumes) had to be flipped before the model could be docked, the poses are in the other hand. Give 2.2 the map as reconstructed (unflipped), then run 2.3: it compares the hands against the particles and writes mirrored poses; you can also force the choice with `HAND`. 3.1 and 4.1 say which poses and model they will use.
* **Inverted contrast.** CryoSTAR expects particles brighter than the background, as RELION and cryoSPARC write them. Stacks for which cryoDRGN needed `--uninvert-data` (particles darker than the background) are the opposite: set `INVERT_CONTRAST` to yes in 2.1 (`auto` does this when every cryoDRGN run on the stack used `--uninvert-data`). 2.3 detects the wrong contrast either way.
* **More than one optics group.** CryoSTAR reads CTF constants from a single optics group; process each group separately.
* **Other settings.** Every option of the original configs can be changed: edit `atom_configs/colab_<RUN_NAME>.py` or `density_configs/colab_<RUN_NAME>.py` under `/content/cryostar/projects/star` (Colab's file browser on the left opens them) and rerun the training cell. The commented `example.py` next to them explains each option. The [CryoSTAR user guide](https://byte-research.gitbook.io/cryostar/) has more detail.

## Citation

If CryoSTAR helps your research, please cite:

> Li Y, Zhou Y, Yuan J, Ye F, Gu Q. CryoSTAR: leveraging structural priors and constraints for cryo-EM heterogeneous reconstruction. *Nature Methods* (2024). [doi:10.1038/s41592-024-02486-1](https://doi.org/10.1038/s41592-024-02486-1)